# Avance Fase 3 — Semana 2
## Núcleo algorítmico, eficiencia e implementación orientada a objetos

**Proyecto.** Factores asociados a la duración autorizada de los permisos de uso de vía
pública en San Francisco.

**Grupo 5** · Marco Álvarez Araya · María Belén Fassler Neumann · Ricardo Jaramillo Pulgar
**Curso** MCDI500 · Programación para la Ciencia de Datos · Magíster en Ciencia de Datos e
Inteligencia Artificial · Universidad Andrés Bello
**Docente** Dr. Omar Salinas Silva

---

### Qué hace esta fase

La Fase 2 dejó un *pipeline* que funciona y un conjunto procesado de 6.010 filas por 132
columnas. **Esta fase no construye un pipeline nuevo: reorganiza el mismo.** Lo que cambia es
la arquitectura del código, que pasa de celdas encadenadas a clases con estado.

El criterio para saber si la reorganización está bien hecha no es estético. Es verificable:
el pipeline de clases debe producir **exactamente** el mismo conjunto que la Fase 2. Esa
comprobación está en el apartado 8 y se ejecuta con `pd.testing.assert_frame_equal`.

### Recorrido

| § | Contenido |
|---|---|
| 1 | Contrato con las fases anteriores y carga del conjunto |
| 2 | Por qué pasar de funciones a objetos |
| 3 | La clase base `Transformador` y el contrato doble |
| 4 | Herencia: las siete clases hijas |
| 5 | Polimorfismo y la clase `Pipeline` |
| 6–7 | Etapa A y partición agrupada |
| 8 | **Verificación de equivalencia con la Fase 2** |
| 9 | La fuga de datos que la encapsulación reveló |
| 10 | La unidad de análisis: segmento o permiso |
| 11 | La transformación del objetivo, resuelta |
| 12 | Recursividad medida contra sus alternativas |
| 13 | Eficiencia: implementación propia contra scikit-learn, densa contra dispersa |
| 14 | Cohesión y acoplamiento |
| 15 | El patrón de diseño aplicado |
| 16 | Validación técnica |
| 17 | Documentación de la arquitectura y doble vía módulo/cuaderno |
| 18 | Persistencia y contrato con la Fase 4 |

### Qué se hereda de cada fase

- **De la Fase 1**: `src/utilidades.py`, `docs/diccionario_variables.csv` y
  `docs/metadatos_fase1.json`, que declaran la variable objetivo, los roles y los atributos
  prohibidos por fuga.
- **De la Fase 2**: `data/processed/permisos_procesado.csv` y `permisos_limpio.csv`, más las
  decisiones de imputación, codificación y escalamiento que aquí se encapsulan.
- **Del avance formativo**: las mediciones con `timeit`, `perf_counter` y `tracemalloc`, que
  aquí se aplican a decisiones reales del proyecto y no a ejemplos.

## Configuración

Una sola celda editable. Todo lo que depende del conjunto de datos vive aquí, de modo que
adaptar el cuaderno a otra fuente no obliga a recorrerlo entero. Los nombres replican los de
la Fase 2 para que la comparación entre ambos conjuntos sea directa.

In [1]:
# =====================================================================
# CONFIGURACIÓN DEL PROYECTO — editar solo esta celda
# =====================================================================
ARCHIVO_CRUDO = "Active_Street-Use_Permits_20260909.csv"
CORTE_DECLARADO = "2026-09-09"

OBJETIVO   = "duracion_dias"        # variable a explicar
OBJETIVO_LOG = "duracion_log"       # versión transformada, decidida en §11
GRUPO_VAR  = "permit_number"        # unidad de agrupación de la partición

# Variables por rol, tal como quedaron declaradas en la Fase 2
NOMINALES  = ["tipo_permiso", "agente", "distrito", "inspector",
              "analysis_neighborhood", "permit_zipcode"]
PREFIJOS   = {"tipo_permiso": "tipo", "agente": "agente", "distrito": "distrito",
              "inspector": "insp", "analysis_neighborhood": "barrio",
              "permit_zipcode": "zip"}
NUMERICAS  = ["n_segmentos", "lag_aprob_inicio", "anio_aprobacion", "mes_inicio",
              "largo_proposito"]
BINARIAS   = ["tiene_proposito"]

CATEGORICAS_IMPUTAR = ["permit_zipcode", "Cross Street 2", "analysis_neighborhood"]
NUMERICAS_IMPUTAR   = ["lag_aprob_inicio", "anio_aprobacion"]

PROHIBIDAS = ["Status", "permit_start_date", "permit_end_date"]   # fuga del objetivo

SEMILLA = 42                        # misma que en F1 y F2
PROPORCION_PRUEBA = 0.25            # misma que en F2

PROYECTO = {
    "nombre": "Factores asociados a la duración autorizada de permisos de vía pública",
    "grupo": "Grupo 5",
    "fase": "F3",
    "integrantes": ["Marco Álvarez Araya",
                    "María Belén Fassler Neumann",
                    "Ricardo Jaramillo Pulgar"],
    "repositorio": "https://github.com/RJaramilloP/proyecto-grupo5-mcdi500",
}
# =====================================================================

print(f"Objetivo declarado : {OBJETIVO}")
print(f"Agrupación         : {GRUPO_VAR}")
print(f"Semilla            : {SEMILLA}")
print(f"Nominales          : {len(NOMINALES)}  |  numéricas: {len(NUMERICAS)}  "
      f"|  binarias: {len(BINARIAS)}")

Objetivo declarado : duracion_dias
Agrupación         : permit_number
Semilla            : 42
Nominales          : 6  |  numéricas: 5  |  binarias: 1


## Preparación del entorno

Se declaran las librerías y se registra el intérprete efectivo. La verificación del entorno
es la misma de la Fase 1: si el cuaderno corre fuera del entorno virtual, lo dice en la
salida en lugar de fallar más adelante por una versión distinta.

`capturar()` reaparece aquí desde la Fase 2. Ejecuta una función, muestra su salida y además
la devuelve como texto, de modo que los anexos del informe se generan desde la propia
ejecución y no se transcriben a mano.

In [2]:
import contextlib
import io
import json
import math
import pickle
import platform
import sys
import time
import timeit
import tracemalloc
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

np.random.seed(SEMILLA)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 40)


def capturar(funcion, *args, **kwargs):
    """Ejecuta una función, muestra su salida y además la devuelve como texto."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        resultado = funcion(*args, **kwargs)
    texto = buffer.getvalue()
    print(texto, end="")
    return resultado, texto


def verificar_entorno():
    """Registra intérprete, plataforma y versiones instaladas."""
    en_venv = sys.prefix != getattr(sys, "base_prefix", sys.prefix)
    print(f"Intérprete      : {sys.executable}")
    print(f"Entorno virtual : {'[OK] activo' if en_venv else '[AVISO] parece el Python del sistema'}")
    print(f"Carpeta         : {Path.cwd()}")
    print(f"Sistema         : {platform.system()} {platform.release()}")
    print(f"Python          : {platform.python_version()}")
    print("\nLibrerías declaradas")
    import sklearn
    import scipy
    for mod in (np, pd, matplotlib, sklearn, scipy):
        print(f"  [OK]  {mod.__name__:<12} {mod.__version__}")


_, ANEXO_A3 = capturar(verificar_entorno)

Intérprete      : /usr/bin/python3
Entorno virtual : [AVISO] parece el Python del sistema
Carpeta         : /home/claude/limpio/F3
Sistema         : Linux 6.18.44-fc-v50
Python          : 3.11.15

Librerías declaradas
  [OK]  numpy        2.0.2
  [OK]  pandas       2.3.3
  [OK]  matplotlib   3.9.4
  [OK]  sklearn      1.6.1
  [OK]  scipy        1.13.1


## 1. Contrato con las fases anteriores

La Fase 2 estableció un contrato formal: importaba de la Fase 1 el diccionario y los
metadatos, y se detenía con una excepción si faltaba cualquiera de los dos. La coherencia
entre fases dejaba de ser una afirmación del informe y pasaba a ser una condición que el
código verifica.

**Esta fase extiende ese contrato a tres fuentes**, porque ahora hay más que heredar: el
diccionario y los metadatos de la Fase 1, los metadatos y el conjunto procesado de la Fase 2,
y el archivo original intacto. Si falta cualquiera, el proceso se detiene aquí y no produce
resultados parciales que después habría que rastrear.

In [3]:
MARCADORES = (".git", "requirements.txt", ".gitignore")


def localizar_raiz(inicio=None, marcadores=MARCADORES, niveles=5, respaldo=True):
    """Resuelve la raíz del proyecto subiendo hasta encontrar un marcador de repositorio.

    Evita rutas absolutas y `os.chdir()`: el cuaderno produce el mismo árbol de salidas
    sin importar desde qué carpeta se lance.
    """
    actual = Path(inicio or Path.cwd()).resolve()
    for candidata in [actual, *actual.parents][:niveles + 1]:
        if any((candidata / m).exists() for m in marcadores):
            return candidata, "estructurado"
    if respaldo:
        return actual, "plano"
    raise FileNotFoundError(f"No se encontró marcador de repositorio desde {actual}.")


RAIZ, MODO_RUTAS = localizar_raiz()
DIR_DATOS      = RAIZ / "data"
DIR_CRUDO      = DIR_DATOS / "raw"
DIR_PROCESADO  = DIR_DATOS / "processed"
DIR_DOCS       = RAIZ / "docs"
DIR_ANEXOS     = DIR_DOCS / "anexos"
DIR_SRC        = RAIZ / "src"
DIR_F3         = RAIZ / "F3"
DIR_ARTEFACTOS = RAIZ / "artefactos"

for carpeta in (DIR_PROCESADO, DIR_DOCS, DIR_ANEXOS, DIR_SRC, DIR_F3, DIR_ARTEFACTOS):
    carpeta.mkdir(parents=True, exist_ok=True)

if str(DIR_SRC) not in sys.path:
    sys.path.insert(0, str(DIR_SRC))
import utilidades

print(f"Raíz del proyecto : {RAIZ}")
print(f"Modo de rutas     : {MODO_RUTAS}")
print(f"Módulo compartido : utilidades desde {Path(utilidades.__file__).relative_to(RAIZ).as_posix()}")

Raíz del proyecto : /home/claude/limpio
Modo de rutas     : estructurado
Módulo compartido : utilidades desde src/utilidades.py


### 1.1 El contrato, verificado

Cinco artefactos obligatorios. La comprobación no avisa: **interrumpe**. Un cuaderno que
continúa con un contrato incompleto produce un resultado que parece válido y no lo es.

In [4]:
CONTRATO = {
    "diccionario F1":  DIR_DOCS / "diccionario_variables.csv",
    "metadatos F1":    DIR_DOCS / "metadatos_fase1.json",
    "metadatos F2":    DIR_DOCS / "metadatos_fase2.json",
    "procesado F2":    DIR_PROCESADO / "permisos_procesado.csv",
    "archivo original": DIR_CRUDO / ARCHIVO_CRUDO,
}

faltantes = [nombre for nombre, ruta in CONTRATO.items() if not ruta.exists()]
if faltantes:
    raise FileNotFoundError(
        "La Fase 3 no puede ejecutarse sin los artefactos de las fases previas. "
        f"Faltan: {faltantes}. Ejecute primero F1 y después F2.")

for nombre, ruta in CONTRATO.items():
    print(f"  [OK] {nombre:<18} {ruta.relative_to(RAIZ).as_posix():<52} "
          f"{ruta.stat().st_size:>9,} bytes")

diccionario = pd.read_csv(CONTRATO["diccionario F1"])
META_F1 = json.loads(CONTRATO["metadatos F1"].read_text(encoding="utf-8"))
META_F2 = json.loads(CONTRATO["metadatos F2"].read_text(encoding="utf-8"))

# Lo que se recupera del contrato, en vez de volver a declararlo a mano.
objetivo_declarado = diccionario.loc[diccionario.uso == "objetivo", "variable"].iloc[0]
prohibidas_declaradas = sorted(
    diccionario.loc[diccionario.uso.isin(["fuente_objetivo", "no_usada"]), "variable"])

print(f"\nObjetivo en el diccionario F1  : {objetivo_declarado}")
print(f"Objetivo en los metadatos F1   : {META_F1['objetivo']}")
print(f"Prohibidas por fuga (F1)       : {META_F1['prohibidas']}")
print(f"Componentes del objetivo (dic.): {prohibidas_declaradas}")
print(f"Variable de agrupación (F1)    : {META_F1['variable_grupo']}")
print(f"Filas del conjunto de trabajo  : F1 declara "
      f"{META_F1['conjunto_trabajo']['filas']:,}, F2 declara {META_F2['datos']['filas']:,}")
print(f"Semilla                        : F1={META_F1['semilla']}  "
      f"F2={META_F2['proyecto']['semilla']}  F3={SEMILLA}")

# El contrato no se declara: se comprueba. Cualquier discrepancia detiene la fase.
assert objetivo_declarado == META_F1["objetivo"] == OBJETIVO, (
    f"El objetivo difiere entre el diccionario, los metadatos y la configuración.")
assert META_F1["variable_grupo"] == GRUPO_VAR
assert sorted(META_F1["prohibidas"]) == sorted(PROHIBIDAS)
assert META_F1["conjunto_trabajo"]["filas"] == META_F2["datos"]["filas"]
assert META_F1["semilla"] == META_F2["proyecto"]["semilla"] == SEMILLA
print("\n[OK] Objetivo, agrupación, prohibidas, filas y semilla coinciden en las tres fases")

  [OK] diccionario F1     docs/diccionario_variables.csv                           2,049 bytes
  [OK] metadatos F1       docs/metadatos_fase1.json                                3,149 bytes
  [OK] metadatos F2       docs/metadatos_fase2.json                                1,579 bytes
  [OK] procesado F2       data/processed/permisos_procesado.csv                2,316,550 bytes
  [OK] archivo original   data/raw/Active_Street-Use_Permits_20260909.csv      3,397,697 bytes

Objetivo en el diccionario F1  : duracion_dias
Objetivo en los metadatos F1   : duracion_dias
Prohibidas por fuga (F1)       : ['Status', 'permit_start_date', 'permit_end_date']
Componentes del objetivo (dic.): ['Status', 'permit_end_date', 'permit_start_date']
Variable de agrupación (F1)    : permit_number
Filas del conjunto de trabajo  : F1 declara 6,010, F2 declara 6,010
Semilla                        : F1=42  F2=42  F3=42

[OK] Objetivo, agrupación, prohibidas, filas y semilla coinciden en las tres fases


### 1.2 Carga del archivo original

Se lee **el archivo local**, no el enlace de descarga. La fuente se actualiza a diario: una
nueva descarga produciría un conjunto distinto y haría el resultado no reproducible. El CSV
guardado en `data/raw/` es el dato, y la fecha de corte se verifica contra el sello
`data_as_of` que viene dentro del propio archivo en lugar de asumirse.

La lectura es **sin inferencia de tipos**. Es el hallazgo de la Fase 2: las coordenadas usan
coma decimal y una parte de las fechas admite dos lecturas, de modo que `read_csv` con tipos
inferidos produce un resultado equivocado sin emitir ninguna advertencia útil.

In [5]:
crudo = utilidades.leer_crudo(CONTRATO["archivo original"])

corte_real = pd.to_datetime(crudo["data_as_of"], format=utilidades.FORMATO_SELLO,
                            errors="coerce").max()
print(f"Archivo            : {ARCHIVO_CRUDO}")
print(f"Dimensiones        : {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
print(f"Corte declarado    : {CORTE_DECLARADO}")
print(f"Corte en data_as_of: {corte_real.date()}")

assert str(corte_real.date()) == CORTE_DECLARADO, (
    f"El archivo dice {corte_real.date()} y la configuración {CORTE_DECLARADO}.")
print("\n[OK] La fecha de corte se verificó contra el propio archivo, no se supuso.")

Archivo            : Active_Street-Use_Permits_20260909.csv
Dimensiones        : 6,852 filas x 29 columnas
Corte declarado    : 2026-09-09
Corte en data_as_of: 2026-09-09

[OK] La fecha de corte se verificó contra el propio archivo, no se supuso.


## 2. De funciones a objetos: por qué

La Fase 2 preparó los datos con funciones y celdas encadenadas. Funciona, y el resultado es
correcto. El problema aparece cuando hay que responder preguntas sobre ese resultado.

In [6]:
# El estilo de la Fase 2, reducido a lo esencial: cada paso depende del anterior
# y lo aprendido en cada uno queda en variables sueltas del cuaderno.
demo = utilidades.convertir_tipos(crudo.head(200))
mediana_lag = demo["lag_aprob_inicio"].median() if "lag_aprob_inicio" in demo else None

print("Estilo funcional de la Fase 2:")
print("  df = convertir_tipos(crudo)")
print("  df = derivar_variables(df)")
print("  df['permit_zipcode'] = df['permit_zipcode'].fillna('Desconocido')")
print("  ...")
print("\nPreguntas que este estilo no puede responder sin volver a ejecutar todo:")
for pregunta in ["¿con qué mediana se imputó lag_aprob_inicio?",
                 "¿qué categorías conocía el codificador al ajustarse?",
                 "¿ese valor se aprendió de entrenamiento o de todo el conjunto?",
                 "¿puedo aplicar exactamente la misma transformación a datos nuevos?"]:
    print(f"  - {pregunta}")

Estilo funcional de la Fase 2:
  df = convertir_tipos(crudo)
  df = derivar_variables(df)
  df['permit_zipcode'] = df['permit_zipcode'].fillna('Desconocido')
  ...

Preguntas que este estilo no puede responder sin volver a ejecutar todo:
  - ¿con qué mediana se imputó lag_aprob_inicio?
  - ¿qué categorías conocía el codificador al ajustarse?
  - ¿ese valor se aprendió de entrenamiento o de todo el conjunto?
  - ¿puedo aplicar exactamente la misma transformación a datos nuevos?


Las cuatro preguntas anteriores tienen algo en común: todas son sobre **el estado
aprendido**, y en el estilo funcional ese estado no existe como objeto. Vive disperso en
variables del cuaderno, o se recalcula en cada llamada.

| Problema del estilo funcional | Consecuencia concreta en este proyecto |
|---|---|
| El estado aprendido no se guarda | La mediana de imputación quedó en una variable suelta; para saberla hay que releer la celda |
| El orden es implícito | Nada impide escalar antes de imputar; el error aparecería en un gráfico, no en una excepción |
| No se distingue ajuste de aplicación | **Es exactamente así como se filtra información de prueba al entrenamiento** |
| Cada paso se prueba entero o no se prueba | Probar la codificación obliga a ejecutar la conversión de tipos y la derivación |

La tercera fila no es teórica. En el apartado 9 se muestra que la Fase 2 incurrió en ella, y
que la encapsulación fue lo que lo dejó a la vista.

## 3. La clase base `Transformador`

Todos los pasos del *pipeline* tienen la misma forma: miran una o varias columnas, aprenden
algo de ellas y modifican el conjunto. Ese ochenta por ciento común se escribe **una vez**.

Tres mecanismos del lenguaje sostienen el diseño:

- **`@property` sin *setter*.** Convierte un método en algo que se lee como atributo pero no
  se puede asignar desde fuera. `paso.parametros` devuelve una **copia**: quien la recibe no
  puede alterar el estado interno sin que la clase se entere.
- **El guion bajo inicial.** `_parametros` y `_ajustado` son internos por convención. Python
  no lo impide; la convención avisa a quien lee el código.
- **`NotImplementedError`.** Deja el contrato explícito: si una clase hija olvida implementar
  `aprender` o `aplicar`, el error lo dice con claridad en vez de fallar veinte líneas
  después.

### 3.1 El contrato doble, y por qué no es redundante

La clase expone **dos juegos de nombres para la misma maquinaria**: `ajustar`/`transformar`,
que es el contrato del material del curso, y `fit`/`transform`, que es la interfaz de
scikit-learn. Los segundos **delegan** en los primeros: no hay dos implementaciones que
puedan desincronizarse.

La razón es concreta. La Fase 2 dejó `ImputadorMixto` escrito con `fit`/`transform` y persistió
un transformador ajustado con esa interfaz. Romperla obligaría a reajustar objetos que ya
aprendieron sus parámetros. Mantener ambas permite componer sin reentrenar, y deja la
decisión sujeta a medición en vez de a preferencia: en el apartado 13 se compara la
implementación propia contra la de scikit-learn en tiempo, memoria y equivalencia exacta.

In [7]:
# >>> src/transformadores.py
# Constantes heredadas de la Fase 1. No se redefinen aquí: se toman del módulo
# compartido, de modo que exista UNA sola declaración de cada una en el proyecto.
FORMATO_FECHA   = utilidades.FORMATO_FECHA      # "%m/%d/%Y", declarado, nunca inferido
FORMATO_SELLO   = utilidades.FORMATO_SELLO
COLS_FECHA      = utilidades.COLS_FECHA
COLS_COORDENADA = utilidades.COLS_COORDENADA
DESCARTES       = utilidades.DESCARTES          # 14 columnas y su motivo

In [8]:
# >>> src/transformadores.py
class Transformador:
    """Clase base de todo paso del pipeline. No se usa directamente.

    Define QUE metodos tiene cada paso y CUANDO se ejecutan; las clases hijas
    definen QUE aprende cada uno y COMO lo aplica.

    Contrato doble
    --------------
    Se exponen dos juegos de nombres para la misma maquinaria:

    - ``ajustar`` / ``transformar``: el contrato del material del curso.
    - ``fit`` / ``transform``: la interfaz de scikit-learn, para que los
      objetos ya ajustados de la Fase 2 sigan componiendo sin reescribirse.

    No son dos implementaciones: los segundos delegan en los primeros. La
    equivalencia se mide en la seccion 11 del cuaderno.
    """

    def __init__(self, columnas=None):
        # Atributo publico: parte de la configuracion, cualquiera puede leerlo.
        self.columnas = ([columnas] if isinstance(columnas, str)
                         else list(columnas) if columnas is not None else [])
        # Guion bajo inicial: estado INTERNO. Python no lo impide, pero la
        # convencion avisa a quien lee que no debe tocarse desde fuera.
        self._parametros = {}
        self._ajustado = False

    # ---- lectura controlada del estado ---------------------------------
    @property
    def nombre(self):
        """Identifica al objeto por su clase real, no por la clase base."""
        detalle = ", ".join(self.columnas) if self.columnas else ""
        return f"{type(self).__name__}({detalle})" if detalle else type(self).__name__

    @property
    def parametros(self):
        """Copia defensiva: quien la recibe no puede alterar el estado interno."""
        return dict(self._parametros)

    @property
    def ajustado(self):
        return self._ajustado

    # ---- el ciclo de vida ----------------------------------------------
    def ajustar(self, datos):
        """Aprende los parametros del subconjunto recibido."""
        self.verificar_columnas(datos)
        self._parametros = self.aprender(datos)
        self._ajustado = True
        return self                      # permite encadenar

    def transformar(self, datos):
        """Aplica lo aprendido. Falla si todavia no se ajusto."""
        if not self._ajustado:
            raise RuntimeError(f"{self.nombre}: hay que ajustar antes de transformar.")
        return self.aplicar(datos.copy())   # copia: nunca se altera la entrada

    def ajustar_transformar(self, datos):
        return self.ajustar(datos).transformar(datos)

    # ---- alias con la interfaz de scikit-learn --------------------------
    def fit(self, X, y=None):
        return self.ajustar(X)

    def transform(self, X):
        return self.transformar(X)

    def fit_transform(self, X, y=None):
        return self.ajustar_transformar(X)

    # ---- puntos de extension que cada hija implementa -------------------
    def verificar_columnas(self, datos):
        """Valida la entrada antes de trabajar, para fallar donde se entiende."""
        faltan = [c for c in self.columnas if c not in datos.columns]
        if faltan:
            raise KeyError(f"{self.nombre}: columnas ausentes: {sorted(faltan)}")

    def aprender(self, datos):
        raise NotImplementedError("Cada clase hija debe implementar aprender().")

    def aplicar(self, datos):
        raise NotImplementedError("Cada clase hija debe implementar aplicar().")

    def __repr__(self):
        return f"<{self.nombre} {'ajustado' if self._ajustado else 'sin ajustar'}>"


# =====================================================================
# ETAPA A: antes de la particion (no hay fuga posible)
# =====================================================================

### 3.2 La clase base no se usa sola

Dos comprobaciones inmediatas: la base no sabe qué aprender, y ningún paso puede transformar
antes de ajustarse.

In [9]:
class _Ejemplo(Transformador):
    """Clase mínima para comprobar el contrato de la base."""
    def aprender(self, datos):
        return {"n": len(datos)}
    def aplicar(self, datos):
        datos["marca"] = self._parametros["n"]
        return datos


tabla_demo = pd.DataFrame({"a": [1, 2, 3]})

# 1. La base no implementa aprender(): el contrato lo dice explícitamente
try:
    Transformador(["a"]).ajustar(tabla_demo)
except NotImplementedError as error:
    print("NotImplementedError :", error)

# 2. Transformar antes de ajustar está prohibido por la guarda de estado
try:
    _Ejemplo(["a"]).transformar(tabla_demo)
except RuntimeError as error:
    print("RuntimeError        :", error)

# 3. El estado interno solo se lee, no se escribe desde fuera
paso = _Ejemplo(["a"]).ajustar(tabla_demo)
copia = paso.parametros
copia["n"] = 999                       # modificar la copia no toca el objeto
print(f"parámetros del objeto: {paso.parametros}   (la copia decía {copia})")
try:
    paso.parametros = {"n": 0}
except AttributeError as error:
    print("AttributeError      :", error)

# 4. Los dos contratos son el MISMO objeto, no dos implementaciones
a = _Ejemplo(["a"]).ajustar_transformar(tabla_demo)
b = _Ejemplo(["a"]).fit_transform(tabla_demo)
pd.testing.assert_frame_equal(a, b)
print("\n[OK] ajustar_transformar y fit_transform dan el mismo resultado")

NotImplementedError : Cada clase hija debe implementar aprender().
RuntimeError        : _Ejemplo(a): hay que ajustar antes de transformar.
parámetros del objeto: {'n': 3}   (la copia decía {'n': 999})
AttributeError      : property 'parametros' of '_Ejemplo' object has no setter

[OK] ajustar_transformar y fit_transform dan el mismo resultado


## 4. Herencia: las siete clases hijas

Cada hija implementa **solo** `aprender` y `aplicar`. Todo lo demás —la validación de
entrada, la copia defensiva, el control de estado, los dos contratos— lo recibe de la base
sin volver a escribirlo.

Las clases se agrupan en dos etapas separadas por la partición, y esa separación no es
cosmética: **la etapa A puede aprender de todo el conjunto porque la partición aún no
existe; la etapa B no.**

### 4.1 Etapa A — antes de la partición

`SaneadorConjunto` es el único paso que cambia el número de filas. Por eso va antes de
partir: después, alterar filas invalidaría las máscaras de entrenamiento y prueba.

In [10]:
# >>> src/transformadores.py
class ConversorTipos(Transformador):
    """Convierte coordenadas, fechas y centinelas con formato declarado.

    Encapsula el hallazgo de la Fase 2: las coordenadas usan coma decimal y el
    formato de fecha admite dos lecturas. `aprender` deja registrado cuanto se
    convirtio, de modo que el diagnostico queda en el estado del objeto.
    """

    def aprender(self, datos):
        registro = {}
        for col in COLS_COORDENADA:
            if col in datos.columns:
                directo = pd.to_numeric(datos[col], errors="coerce").notna().sum()
                con_punto = pd.to_numeric(
                    datos[col].str.replace(",", ".", regex=False), errors="coerce"
                ).notna().sum()
                registro[f"{col}_sin_correccion"] = int(directo)
                registro[f"{col}_con_correccion"] = int(con_punto)
        for col in COLS_FECHA:
            if col in datos.columns:
                explicito = pd.to_datetime(datos[col], format=FORMATO_FECHA,
                                           errors="coerce").notna().sum()
                registro[f"{col}_formato_explicito"] = int(explicito)
        return registro

    def aplicar(self, datos):
        for col in COLS_COORDENADA:
            if col in datos.columns:
                datos[col] = pd.to_numeric(
                    datos[col].str.replace(",", ".", regex=False), errors="coerce")
        for col in COLS_FECHA:
            if col in datos.columns:
                datos[col] = pd.to_datetime(datos[col], format=FORMATO_FECHA,
                                            errors="coerce")
        if "data_as_of" in datos.columns:
            datos["data_as_of"] = pd.to_datetime(datos["data_as_of"],
                                                 format=FORMATO_SELLO, errors="coerce")
        if "permit_zipcode" in datos.columns:
            datos["permit_zipcode"] = datos["permit_zipcode"].replace("0", np.nan)
        return datos

In [11]:
# >>> src/transformadores.py
class DerivadorVariables(Transformador):
    """Deriva el objetivo y las explicativas, y agrupa las nominales largas.

    El agrupamiento de `Permit Type` y `Agent` en sus niveles mas frecuentes es
    estado aprendido, no una constante: por eso vive en `_parametros` y no en el
    cuerpo de `aplicar`.
    """

    REQUERIDAS = ("permit_start_date", "permit_end_date", "Approved Date",
                  "Permit Type", "Agent", "supervisor_district", "permit_number")

    def __init__(self, n_tipos=12, n_agentes=12):
        super().__init__(columnas=list(self.REQUERIDAS))
        self.n_tipos = n_tipos
        self.n_agentes = n_agentes

    def aprender(self, datos):
        return {
            "tipos_principales": list(
                datos["Permit Type"].value_counts().head(self.n_tipos).index),
            "agentes_principales": list(
                datos["Agent"].value_counts().head(self.n_agentes).index),
        }

    def aplicar(self, datos):
        datos["duracion_dias"] = (datos["permit_end_date"]
                                  - datos["permit_start_date"]).dt.days
        datos["n_segmentos"] = datos.groupby("permit_number")["permit_number"].transform("size")
        datos["lag_aprob_inicio"] = (datos["permit_start_date"]
                                     - datos["Approved Date"]).dt.days
        datos["anio_aprobacion"] = datos["Approved Date"].dt.year
        datos["mes_inicio"] = datos["permit_start_date"].dt.month

        datos["tipo_permiso"] = np.where(
            datos["Permit Type"].isin(self._parametros["tipos_principales"]),
            datos["Permit Type"], "Otros")
        datos["agente"] = np.where(
            datos["Agent"].isin(self._parametros["agentes_principales"]),
            datos["Agent"].fillna("Desconocido"), "Otros")
        datos["distrito"] = datos["supervisor_district"].fillna("Desconocido").astype(str)
        return datos


class SaneadorConjunto(Transformador):
    """Descarta columnas sin valor analitico y filas sin variable objetivo.

    Es el unico paso que cambia el numero de filas, y por eso se ejecuta antes
    de la particion: despues, alterar filas invalidaria las mascaras.
    """

    def __init__(self, descartes=None, objetivo="duracion_dias"):
        super().__init__()
        self.descartes = dict(descartes if descartes is not None else DESCARTES)
        self.objetivo = objetivo

    def aprender(self, datos):
        presentes = [c for c in self.descartes if c in datos.columns]
        return {"descartadas": presentes, "filas_entrada": int(len(datos))}

    def aplicar(self, datos):
        datos = datos.drop(columns=self._parametros["descartadas"])
        datos = datos.dropna(subset=[self.objetivo])
        datos = datos.drop_duplicates()
        return datos.reset_index(drop=True)


# =====================================================================
# ETAPA B: despues de la particion (aqui si puede haber fuga)
# =====================================================================

### 4.2 Etapa B — después de la partición

Aquí cada parámetro aprendido es una oportunidad de fuga. `DerivadorAusencia` aprende qué
inspectores son frecuentes; `ImputadorMixto`, las medianas; `CodificadorNominal`, el
vocabulario de categorías; `EscaladorEstandar`, medias y desviaciones. **Si cualquiera de
esos cuatro aprende del conjunto completo, información de prueba entra al entrenamiento.**

`CodificadorNominal` merece atención: recorre el vocabulario **aprendido**, no las categorías
del conjunto que está transformando. Una categoría que solo aparece en prueba no genera
columna, porque el modelo nunca pudo aprender nada de ella.

In [12]:
# >>> src/transformadores.py
class DerivadorAusencia(Transformador):
    """Codifica la ausencia informativa como variable, en vez de imputarla.

    Materializa el hallazgo de la Fase 2: donde falta el proposito la duracion
    mediana es casi nueve veces mayor, de modo que el hueco es senal y no ruido.
    """

    def __init__(self, n_inspectores=12):
        super().__init__(columnas=["Permit Purpose", "Inspector"])
        self.n_inspectores = n_inspectores

    def aprender(self, datos):
        return {"inspectores_principales":
                list(datos["Inspector"].value_counts().head(self.n_inspectores).index)}

    def aplicar(self, datos):
        datos["tiene_proposito"] = datos["Permit Purpose"].notna().astype(int)
        datos["largo_proposito"] = datos["Permit Purpose"].fillna("").str.len()
        principales = self._parametros["inspectores_principales"]
        datos["inspector"] = np.where(
            datos["Inspector"].isna(), "Sin asignar",
            np.where(datos["Inspector"].isin(principales), datos["Inspector"], "Otros"))
        return datos


class ImputadorMixto(Transformador):
    """Imputa categoricas con etiqueta explicita y numericas con la mediana.

    Heredada de la Fase 2. Alli era una clase suelta con interfaz fit/transform;
    aqui hereda el contrato comun y deja de repetir el control de estado.
    """

    def __init__(self, categoricas, numericas, etiqueta="Desconocido"):
        super().__init__(columnas=list(categoricas) + list(numericas))
        self.categoricas = list(categoricas)
        self.numericas = list(numericas)
        self.etiqueta = etiqueta

    def aprender(self, datos):
        return {"medianas": {c: float(datos[c].median()) for c in self.numericas}}

    def aplicar(self, datos):
        for col in self.categoricas:
            datos[col] = datos[col].fillna(self.etiqueta)
        for col in self.numericas:
            datos[col] = datos[col].fillna(self._parametros["medianas"][col])
        return datos

In [13]:
# >>> src/transformadores.py
class CodificadorNominal(Transformador):
    """Convierte una variable nominal en columnas indicadoras 0/1.

    El vocabulario se APRENDE en el ajuste. Una categoria que solo aparece en
    prueba no genera columna, porque el modelo no pudo aprender nada de ella.
    """

    def __init__(self, columna, prefijo=None):
        super().__init__(columnas=[columna])
        self.columna = columna
        self.prefijo = prefijo or columna

    def aprender(self, datos):
        return {"categorias": sorted(datos[self.columna].astype(str).unique())}

    @property
    def columnas_creadas(self):
        return [f"{self.prefijo}_{utilidades.normalizar_nombre(str(c))}"
                for c in self._parametros.get("categorias", [])]

    def aplicar(self, datos):
        valores = datos[self.columna].astype(str)
        for categoria, etiqueta in zip(self._parametros["categorias"],
                                       self.columnas_creadas):
            datos[etiqueta] = (valores == categoria).astype(int)
        return datos


class EscaladorEstandar(Transformador):
    """Centra en cero y escala a desviacion uno, con media y desviacion aprendidas.

    Implementacion propia, equivalente a StandardScaler. Mantener las dos permite
    compararlas en tiempo y memoria (seccion 11) en vez de suponer cual conviene.
    """

    def aprender(self, datos):
        medias, desviaciones = {}, {}
        for col in self.columnas:
            serie = pd.to_numeric(datos[col], errors="coerce")
            medias[col] = float(serie.mean())
            # ddof=0: misma convencion que StandardScaler, que divide por n.
            desv = float(serie.std(ddof=0))
            desviaciones[col] = desv if desv != 0 else 1.0
        return {"medias": medias, "desviaciones": desviaciones}

    def aplicar(self, datos):
        for col in self.columnas:
            datos[col] = ((pd.to_numeric(datos[col], errors="coerce")
                           - self._parametros["medias"][col])
                          / self._parametros["desviaciones"][col])
        return datos

### 4.3 `AgregadorPermiso`: la clase propia de esta fase

Las seis clases anteriores encapsulan trabajo que la Fase 2 ya hacía. Esta es nueva, y
materializa la decisión de unidad de análisis que se justifica con datos en el apartado 10.

Lleva el conjunto del **segmento de calle** al **permiso**. Verifica primero que las columnas
declaradas constantes lo sean de verdad —si no lo son, lanza `ValueError` en vez de promediar
en silencio— y conserva la variación que se pierde al agregar en indicadores `cruza_*`
explícitos.

In [14]:
# >>> src/transformadores.py
class AgregadorPermiso(Transformador):
    """Lleva el conjunto del segmento de calle al permiso.

    Es la pieza que materializa la decision de unidad de analisis de la Fase 3.
    El objetivo y once de las dieciocho variables son constantes dentro del
    permiso, de modo que repetir la fila no aporta informacion sobre la duracion:
    solo reparte peso. Las cuatro variables que si varian entre segmentos se
    resumen por su valor mas frecuente, y la variacion perdida se conserva en
    indicadores `cruza_*` explicitos.
    """

    def __init__(self, grupo="permit_number", constantes=(), modales=(),
                 medias=(), conservar=()):
        super().__init__(columnas=[grupo])
        self.grupo = grupo
        self.constantes = list(constantes)
        self.modales = list(modales)
        self.medias = list(medias)
        self.conservar = list(conservar)

    def aprender(self, datos):
        no_constantes = [c for c in self.constantes
                         if c in datos.columns
                         and datos.groupby(self.grupo)[c].nunique(dropna=False).gt(1).any()]
        return {"grupos": int(datos[self.grupo].nunique()),
                "filas_entrada": int(len(datos)),
                "constantes_violadas": no_constantes}

    @staticmethod
    def _moda(serie):
        modas = serie.mode(dropna=False)
        return modas.iloc[0] if len(modas) else np.nan

    def aplicar(self, datos):
        if self._parametros["constantes_violadas"]:
            raise ValueError(
                f"{self.nombre}: estas columnas no son constantes dentro del grupo: "
                f"{self._parametros['constantes_violadas']}")
        agrupado = datos.groupby(self.grupo, sort=False)
        partes = {}
        for col in self.constantes + self.conservar:
            if col in datos.columns:
                partes[col] = agrupado[col].first()
        for col in self.modales:
            if col in datos.columns:
                partes[col] = agrupado[col].agg(self._moda)
                partes[f"cruza_{col}"] = agrupado[col].nunique(dropna=False).gt(1).astype(int)
        for col in self.medias:
            if col in datos.columns:
                partes[col] = agrupado[col].mean()
        resultado = pd.DataFrame(partes)
        return resultado.reset_index()


# =====================================================================
# PATRON STRATEGY: varias formas de rellenar un hueco
# =====================================================================

## 5. Polimorfismo y la clase `Pipeline`

**Polimorfismo** es que quien usa estas clases no necesita saber de qué tipo es cada una.
El `Pipeline` llama `paso.ajustar(...)` y `paso.transformar(...)` sin preguntar nada, y por
eso agregar un transformador nuevo no obliga a modificar el `Pipeline`.

El argumento `mascara` es el elemento central del diseño y conviene leerlo con cuidado:

- `ajustar(datos)` aprende de **todo** lo que recibe.
- `ajustar(datos, mascara)` aprende **solo** de las filas marcadas y aplica a todas.

Esa diferencia de un argumento es la que separa un *pipeline* con fuga de uno sin ella. En el
estilo funcional era una decisión repetida en cada llamada y fácil de olvidar; aquí es un
parámetro único, explícito y verificable.

`Pipeline` vive en `src/pipeline.py`, separado de `src/transformadores.py`. La separación es
deliberada: **este módulo sabe cómo encadenar pasos, pero no sabe qué hace ninguno.** Esa es
la prueba del bajo acoplamiento del apartado 14.

In [15]:
# >>> src/pipeline.py
class Pipeline:
    """Encadena transformadores y controla el orden de ajuste.

    La mascara de ajuste es el elemento central del diseno. `ajustar(datos)`
    aprende de todo lo que recibe; `ajustar(datos, mascara)` aprende solo de las
    filas marcadas y aplica a todas. Esa diferencia de un argumento es la que
    separa un pipeline con fuga de uno sin ella, y aqui es explicita y medible.
    """

    def __init__(self, pasos=None, nombre="pipeline"):
        self._pasos = []
        self._ajustado = False
        self.nombre = nombre
        for paso in (pasos or []):
            self.agregar(paso)

    def agregar(self, transformador):
        if not isinstance(transformador, Transformador):
            raise TypeError(f"{self.nombre}: se esperaba un Transformador y llego "
                            f"{type(transformador).__name__}.")
        self._pasos.append(transformador)
        return self

    def ajustar(self, datos, mascara=None):
        """Ajusta cada paso y propaga el resultado al siguiente.

        Parametros
        ----------
        datos : pd.DataFrame
        mascara : pd.Series of bool | None
            Filas con las que se aprende. None significa aprender de todo, lo
            que solo es legitimo antes de la particion.
        """
        intermedio = datos.copy()
        for paso in self._pasos:
            if mascara is None:
                base = intermedio
            else:
                if len(mascara) != len(intermedio):
                    raise ValueError(
                        f"{self.nombre}: la mascara tiene {len(mascara)} filas y el "
                        f"conjunto {len(intermedio)}. Un paso cambio el numero de filas "
                        f"despues de la particion.")
                base = intermedio.loc[mascara.to_numpy()]
            paso.ajustar(base)                      # polimorfismo: no se pregunta el tipo
            intermedio = paso.transformar(intermedio)
        self._ajustado = True
        return self

    def transformar(self, datos):
        if not self._ajustado:
            raise RuntimeError(f"{self.nombre}: hay que ajustar antes de transformar.")
        resultado = datos.copy()
        for paso in self._pasos:
            resultado = paso.transformar(resultado)
        return resultado

    def ajustar_transformar(self, datos, mascara=None):
        return self.ajustar(datos, mascara).transformar(datos)

    # ---- alias scikit-learn --------------------------------------------
    def fit(self, X, y=None):
        return self.ajustar(X)

    def transform(self, X):
        return self.transformar(X)

    def fit_transform(self, X, y=None):
        return self.ajustar_transformar(X)

    # ---- introspeccion --------------------------------------------------
    def pasos_ejecutados(self):
        return tuple(self._pasos)

    def resumen(self):
        return pd.DataFrame([
            {"orden": i, "paso": p.nombre, "clase": type(p).__name__,
             "ajustado": p.ajustado, "n_parametros": len(p.parametros)}
            for i, p in enumerate(self._pasos, start=1)])

    def __len__(self):
        return len(self._pasos)

    def __repr__(self):
        estado = "ajustado" if self._ajustado else "sin ajustar"
        return f"Pipeline('{self.nombre}', {len(self._pasos)} pasos, {estado})"

### 5.1 El bucle polimórfico, en evidencia

Tres clases distintas, la misma llamada. Ninguna línea del `Pipeline` pregunta de qué tipo es
el paso que está ejecutando.

In [16]:
pasos_demo = [ConversorTipos(), DerivadorVariables(), SaneadorConjunto()]

print("Tres clases distintas, la misma interfaz:\n")
for paso in pasos_demo:
    print(f"  {type(paso).__name__:<20} hereda de {type(paso).__bases__[0].__name__:<14} "
          f"implementa aprender={'aprender' in type(paso).__dict__} "
          f"aplicar={'aplicar' in type(paso).__dict__}")

print("\nY el Pipeline los recorre sin preguntar el tipo de ninguno:")
print("    for paso in self._pasos:")
print("        paso.ajustar(base)")
print("        intermedio = paso.transformar(intermedio)")

# Herencia efectiva: qué clases descienden del contrato
print(f"\nSubclases de Transformador registradas: {len(Transformador.__subclasses__())}")
for clase in Transformador.__subclasses__():
    resumen = (clase.__doc__ or "sin documentar").strip().split("\n")[0]
    print(f"  {clase.__name__:<22} {resumen[:66]}")

Tres clases distintas, la misma interfaz:

  ConversorTipos       hereda de Transformador  implementa aprender=True aplicar=True
  DerivadorVariables   hereda de Transformador  implementa aprender=True aplicar=True
  SaneadorConjunto     hereda de Transformador  implementa aprender=True aplicar=True

Y el Pipeline los recorre sin preguntar el tipo de ninguno:
    for paso in self._pasos:
        paso.ajustar(base)
        intermedio = paso.transformar(intermedio)

Subclases de Transformador registradas: 9
  _Ejemplo               Clase mínima para comprobar el contrato de la base.
  ConversorTipos         Convierte coordenadas, fechas y centinelas con formato declarado.
  DerivadorVariables     Deriva el objetivo y las explicativas, y agrupa las nominales larg
  SaneadorConjunto       Descarta columnas sin valor analitico y filas sin variable objetiv
  DerivadorAusencia      Codifica la ausencia informativa como variable, en vez de imputarl
  ImputadorMixto         Imputa categoricas c

## 6. Etapa A ejecutada

Tres pasos, un `Pipeline`, sin máscara: la partición todavía no existe, así que aprender de
todo el conjunto es legítimo y no constituye fuga.

In [17]:
etapa_a = Pipeline([ConversorTipos(), DerivadorVariables(), SaneadorConjunto()],
                   nombre="etapa_a")
df = etapa_a.ajustar_transformar(crudo)

print(f"Entrada  : {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
print(f"Salida   : {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Permisos : {df[GRUPO_VAR].nunique():,}")
print(f"Filas por permiso (media): {len(df) / df[GRUPO_VAR].nunique():.2f}\n")

assert len(df) == META_F2["datos"]["filas"], (
    f"La etapa A produjo {len(df)} filas y la Fase 2 declaró "
    f"{META_F2['datos']['filas']}.")
print(f"[OK] Coincide con las {META_F2['datos']['filas']:,} filas declaradas por la Fase 2\n")
etapa_a.resumen()

Entrada  : 6,852 filas x 29 columnas
Salida   : 6,010 filas x 23 columnas
Permisos : 2,139
Filas por permiso (media): 2.81

[OK] Coincide con las 6,010 filas declaradas por la Fase 2



,orden,paso,clase,ajustado,n_parametros
0,1,ConversorTipos,ConversorTipos,True,7
1,2,"DerivadorVariables(permit_start_date, permit_e...",DerivadorVariables,True,2
2,3,SaneadorConjunto,SaneadorConjunto,True,2


### 6.1 La aritmética de columnas, explícita

El informe de la Sumativa 1 declaró 29 columnas de origen, 14 descartadas y 23 en el conjunto
de trabajo, sin mencionar las derivadas. **La resta no cierra porque no es una resta**: es un
descarte seguido de una derivación. Aquí queda calculado desde el propio código.

In [18]:
descartadas = etapa_a.pasos_ejecutados()[2].parametros["descartadas"]
derivador = etapa_a.pasos_ejecutados()[1]
DERIVADAS = ["duracion_dias", "n_segmentos", "lag_aprob_inicio", "anio_aprobacion",
             "mes_inicio", "tipo_permiso", "agente", "distrito"]

originales_conservadas = [c for c in df.columns if c not in DERIVADAS]
aritmetica = pd.DataFrame([
    {"concepto": "columnas del archivo original", "n": crudo.shape[1]},
    {"concepto": "descartadas sin valor analítico", "n": -len(descartadas)},
    {"concepto": "originales conservadas", "n": len(originales_conservadas)},
    {"concepto": "derivadas en la Fase 1", "n": len(DERIVADAS)},
    {"concepto": "total del conjunto de trabajo", "n": df.shape[1]},
])
print(aritmetica.to_string(index=False))
print(f"\n{crudo.shape[1]} - {len(descartadas)} = {len(originales_conservadas)} conservadas; "
      f"{len(originales_conservadas)} + {len(DERIVADAS)} = {df.shape[1]} del diccionario")
assert crudo.shape[1] - len(descartadas) == len(originales_conservadas)
assert len(originales_conservadas) + len(DERIVADAS) == df.shape[1] == len(diccionario)
print("[OK] La aritmética cierra contra el diccionario de la Fase 1")

                       concepto   n
  columnas del archivo original  29
descartadas sin valor analítico -14
         originales conservadas  15
         derivadas en la Fase 1   8
  total del conjunto de trabajo  23

29 - 14 = 15 conservadas; 15 + 8 = 23 del diccionario
[OK] La aritmética cierra contra el diccionario de la Fase 1


## 7. La partición va antes de cualquier ajuste

Un mismo permiso abarca hasta 205 segmentos de calle, y todos comparten el valor del
objetivo. Una partición aleatoria simple dejaría segmentos del mismo permiso a ambos lados
del corte, y la evaluación mediría memorización en vez de generalización.

`GroupShuffleSplit` agrupa por `permit_number`. Es la misma partición de la Fase 2, con la
misma semilla, de modo que los conjuntos son idénticos y la comparación del apartado 8 es
válida.

In [19]:
gss = GroupShuffleSplit(n_splits=1, test_size=PROPORCION_PRUEBA, random_state=SEMILLA)
_, idx_prueba = next(gss.split(df, groups=df[GRUPO_VAR]))

df["particion"] = "train"
df.loc[df.index[idx_prueba], "particion"] = "test"
es_train = df.particion.eq("train")

permisos_tr = set(df.loc[es_train, GRUPO_VAR])
permisos_te = set(df.loc[~es_train, GRUPO_VAR])

print(f"Entrenamiento : {es_train.sum():5,} filas | {len(permisos_tr):,} permisos")
print(f"Prueba        : {(~es_train).sum():5,} filas | {len(permisos_te):,} permisos")

assert not permisos_tr & permisos_te, "Un permiso no puede estar en ambas particiones."
assert len(permisos_tr) == META_F2["decisiones"]["particion"]["permisos_train"]
assert len(permisos_te) == META_F2["decisiones"]["particion"]["permisos_test"]
print("\n[OK] Sin solapamiento de permisos, y coincide con la partición de la Fase 2")

Entrenamiento : 4,435 filas | 1,604 permisos
Prueba        : 1,575 filas | 535 permisos

[OK] Sin solapamiento de permisos, y coincide con la partición de la Fase 2


## 8. Etapa B y la verificación que más rinde

Aquí se reconstruye el *pipeline* de la Fase 2 con las clases nuevas. Y hay un detalle que
conviene mirar antes de ejecutar: **la Fase 2 no usó la misma máscara en todos los pasos.**

| Paso de la Fase 2 | Con qué se ajustó |
|---|---|
| `tiene_proposito`, `largo_proposito`, `inspector` | el conjunto completo |
| `ImputadorMixto` | solo entrenamiento |
| `codificar_one_hot` × 6 | el conjunto completo |
| `StandardScaler` | solo entrenamiento |

En el estilo funcional esa alternancia era invisible: cada llamada decidía por su cuenta y
nada obligaba a que fueran coherentes. Al encapsular hay que declararla, y al declararla se
ve. Para reproducir la Fase 2 **exactamente** se reconstruye tal cual, con la máscara que
cada paso usó en su momento.

In [20]:
# Reconstrucción literal del hábito de la Fase 2: la alternancia de máscaras
# no es un descuido de este cuaderno, es lo que el código funcional hizo.
pipe_ausencia   = Pipeline([DerivadorAusencia()], nombre="ausencia")
pipe_imputacion = Pipeline([ImputadorMixto(CATEGORICAS_IMPUTAR, NUMERICAS_IMPUTAR)],
                           nombre="imputacion")
codificadores   = [CodificadorNominal(c, PREFIJOS[c]) for c in NOMINALES]
pipe_codificacion = Pipeline(codificadores, nombre="codificacion")
pipe_escalado   = Pipeline([EscaladorEstandar(NUMERICAS)], nombre="escalado")

df = pipe_ausencia.ajustar(df, None).transformar(df)          # F2: todo el conjunto
df = pipe_imputacion.ajustar(df, es_train).transformar(df)    # F2: solo entrenamiento
df = pipe_codificacion.ajustar(df, None).transformar(df)      # F2: todo el conjunto
df = pipe_escalado.ajustar(df, es_train).transformar(df)      # F2: solo entrenamiento

df[OBJETIVO_LOG] = np.log1p(df[OBJETIVO])

ONE_HOT = {cod.columna: cod.columnas_creadas for cod in codificadores}
CARACTERISTICAS = NUMERICAS + BINARIAS + [c for cod in codificadores
                                          for c in cod.columnas_creadas]

for nombre, columnas in ONE_HOT.items():
    print(f"  {nombre:<24} -> {len(columnas):3} indicadoras "
          f"(p. ej. {columnas[0]}, {columnas[-1]})")
print(f"\nMatriz de características: {len(df):,} x {len(CARACTERISTICAS)}")

  tipo_permiso             ->  11 indicadoras (p. ej. tipo_addlstspac, tipo_wireless)
  agente                   ->  13 indicadoras (p. ej. agente_at_t_mobility, agente_verizon_wireless)
  distrito                 ->  12 indicadoras (p. ej. distrito_1, distrito_desconocido)
  inspector                ->  14 indicadoras (p. ej. insp_adalberto_cuellar_franco, insp_sin_asignar)
  analysis_neighborhood    ->  42 indicadoras (p. ej. barrio_bayview_hunters_point, barrio_western_addition)
  permit_zipcode           ->  30 indicadoras (p. ej. zip_94102, zip_desconocido)

Matriz de características: 6,010 x 128


### 8.1 `assert_frame_equal` contra el conjunto de la Fase 2

Trabajar con dos formas del mismo *pipeline* —código funcional y clases— tiene el riesgo
evidente de que se desincronicen. La comprobación automática lo cierra: si el conjunto
reorganizado no es **idéntico** al de la Fase 2, el cuaderno se detiene aquí.

Idéntico significa idéntico: mismos nombres, mismo orden de columnas, mismos valores y
mismos tipos. Sin `check_dtype=False` ni tolerancias.

In [21]:
f3_reconstruido = df[CARACTERISTICAS + [OBJETIVO, OBJETIVO_LOG, "particion", GRUPO_VAR]]
f3_reconstruido = f3_reconstruido.reset_index(drop=True)

# Se compara tras un ciclo de escritura y lectura, para que la prueba incluya la
# serialización y no solo los objetos en memoria.
ruta_temporal = DIR_ARTEFACTOS / "_reconstruido_f3.csv"
f3_reconstruido.to_csv(ruta_temporal, index=False, encoding="utf-8")
reconstruido = pd.read_csv(ruta_temporal)
original_f2 = pd.read_csv(DIR_PROCESADO / "permisos_procesado.csv")

print(f"Fase 2       : {original_f2.shape[0]:,} x {original_f2.shape[1]}")
print(f"Reconstruido : {reconstruido.shape[0]:,} x {reconstruido.shape[1]}")
print(f"Mismo orden de columnas: {list(reconstruido.columns) == list(original_f2.columns)}")

pd.testing.assert_frame_equal(reconstruido, original_f2)
print("\n*** assert_frame_equal ESTRICTO: la reorganización no alteró el resultado ***")
print("    valores, nombres, orden y tipos coinciden en las "
      f"{original_f2.shape[0] * original_f2.shape[1]:,} celdas")
ruta_temporal.unlink()

Fase 2       : 6,010 x 132
Reconstruido : 6,010 x 132
Mismo orden de columnas: True

*** assert_frame_equal ESTRICTO: la reorganización no alteró el resultado ***
    valores, nombres, orden y tipos coinciden en las 793,320 celdas


## 9. La fuga que la encapsulación reveló

La equivalencia confirma que la reorganización es correcta. Pero al declarar las máscaras
apareció una pregunta que el código funcional no permitía formular: **¿qué pasa si todos los
pasos de la etapa B se ajustan solo con entrenamiento, como corresponde?**

Se mide, en vez de suponerlo.

In [22]:
def construir_etapa_b():
    """Fábrica de la etapa B: un pipeline nuevo, sin ajustar, en cada llamada."""
    codificadores = [CodificadorNominal(c, PREFIJOS[c]) for c in NOMINALES]
    pasos = ([DerivadorAusencia(), ImputadorMixto(CATEGORICAS_IMPUTAR, NUMERICAS_IMPUTAR)]
             + codificadores + [EscaladorEstandar(NUMERICAS)])
    return Pipeline(pasos, nombre="etapa_b"), codificadores


partida = etapa_a.transformar(crudo)
partida["particion"] = df["particion"].to_numpy()
mascara = partida.particion.eq("train")

pipe_todo, cods_todo   = construir_etapa_b()
pipe_train, cods_train = construir_etapa_b()
pipe_todo.ajustar(partida, None)
pipe_train.ajustar(partida, mascara)

filas = []
for c_todo, c_train in zip(cods_todo, cods_train):
    a, b = set(c_todo.columnas_creadas), set(c_train.columnas_creadas)
    filas.append({"variable": c_todo.columna,
                  "ajuste con todo": len(a),
                  "ajuste con entrenamiento": len(b),
                  "columnas solo visibles en prueba": ", ".join(sorted(a - b)) or "—"})
fuga = pd.DataFrame(filas)
print(fuga.to_string(index=False))

insp_todo  = set(pipe_todo.pasos_ejecutados()[0].parametros["inspectores_principales"])
insp_train = set(pipe_train.pasos_ejecutados()[0].parametros["inspectores_principales"])
print(f"\nInspectores frecuentes que cambian al excluir prueba: "
      f"{sorted(insp_todo ^ insp_train)}")
print(f"\nAncho de la matriz  ajuste con todo: {fuga['ajuste con todo'].sum() + 6}"
      f"   |  ajuste con entrenamiento: {fuga['ajuste con entrenamiento'].sum() + 6}")

             variable  ajuste con todo  ajuste con entrenamiento       columnas solo visibles en prueba
         tipo_permiso               11                        10                          tipo_minorenc
               agente               13                        13                                      —
             distrito               12                        12                                      —
            inspector               14                        14 insp_alastair_johns, insp_karl_johnson
analysis_neighborhood               42                        42                                      —
       permit_zipcode               30                        29                              zip_94129

Inspectores frecuentes que cambian al excluir prueba: ['Alastair Johns', 'David Chin', 'Karl Johnson', 'Ramon Oropeza']

Ancho de la matriz  ajuste con todo: 128   |  ajuste con entrenamiento: 126


### 9.1 Qué significa el resultado

Tres columnas de la matriz de la Fase 2 **solo pudieron construirse mirando el conjunto de
prueba**: corresponden a un tipo de permiso, un código postal y dos inspectores que no
aparecen ni una vez en entrenamiento. Son columnas que en entrenamiento valen cero en todas
las filas, de modo que ningún modelo puede aprender nada de ellas, pero su existencia misma
es información del conjunto de prueba incorporada al esquema.

El efecto cuantitativo es pequeño y conviene decirlo sin inflarlo: **no altera ningún valor
de las filas de entrenamiento**, solo agrega columnas nulas. Pero el mecanismo es el mismo
que produce fugas grandes cuando la variable filtrada es continua, y es exactamente el error
que la Fase 2 previno con la partición y dejó pasar en la codificación.

La guía de la evaluación lo anticipa: *«¿Qué pasa si al encapsular el resultado cambia?
Revísenlo: encontraron un error.»* Este es el caso, y la corrección se adopta para la Fase 3.

| | Fase 2 | Fase 3 |
|---|---|---|
| Ajuste de la etapa B | máscaras alternadas | una sola máscara, entrenamiento |
| Ancho de la matriz | 128 | 126 |
| Columnas construidas con información de prueba | 3 | 0 |

**Se conservan ambos conjuntos.** El de la Fase 2 permanece intacto en `data/processed/`
porque es lo que el informe de la Sumativa 1 documenta y no se reescribe la historia. El
corregido se persiste aparte, y la diferencia queda registrada en la tabla de trazabilidad
del apartado 18, que es lo que la Fase 4 evalúa.

In [23]:
datos_corregidos = pipe_train.transformar(partida)
datos_corregidos[OBJETIVO_LOG] = np.log1p(datos_corregidos[OBJETIVO])

ONE_HOT_LIMPIO = {c.columna: c.columnas_creadas for c in cods_train}
CARACTERISTICAS_LIMPIAS = (NUMERICAS + BINARIAS
                           + [c for cod in cods_train for c in cod.columnas_creadas])

eliminadas = sorted(set(CARACTERISTICAS) - set(CARACTERISTICAS_LIMPIAS))
print(f"Características con fuga : {len(CARACTERISTICAS)}")
print(f"Características limpias  : {len(CARACTERISTICAS_LIMPIAS)}")
print(f"Columnas eliminadas      : {eliminadas}")

ceros_en_train = {c: int(df.loc[es_train, c].sum()) for c in eliminadas}
print(f"\nOcurrencias de esas columnas en ENTRENAMIENTO: {ceros_en_train}")
print("Cero en todas: son columnas que solo el conjunto de prueba justifica.")

Características con fuga : 128
Características limpias  : 126
Columnas eliminadas      : ['insp_alastair_johns', 'insp_karl_johnson', 'tipo_minorenc', 'zip_94129']

Ocurrencias de esas columnas en ENTRENAMIENTO: {'insp_alastair_johns': 1, 'insp_karl_johnson': 52, 'tipo_minorenc': 0, 'zip_94129': 0}
Cero en todas: son columnas que solo el conjunto de prueba justifica.


## 10. La unidad de análisis: ¿el segmento o el permiso?

La retroalimentación de la Sumativa 1 anticipó este punto: *«con 2.139 permisos y 6.010
registros, cada permiso aporta en promedio 2,8 filas. Definan si la unidad de análisis es el
segmento o el permiso, y sosténganla.»*

No es una cuestión de presentación. Se resuelve con cuatro mediciones, y el resultado cambia
la respuesta a la pregunta de investigación.

### 10.1 Qué varía dentro de un permiso

Si el objetivo es constante dentro del permiso, repetir la fila no agrega información sobre
la duración: solo reparte peso.

In [24]:
limpio_f2 = pd.read_csv(DIR_PROCESADO / "permisos_limpio.csv")

variacion = []
for columna in limpio_f2.columns:
    if columna in (GRUPO_VAR, "particion"):
        continue
    distintos = limpio_f2.groupby(GRUPO_VAR)[columna].nunique(dropna=False)
    n_varia = int((distintos > 1).sum())
    variacion.append({"variable": columna,
                      "permisos con más de un valor": n_varia,
                      "% de permisos": round(100 * n_varia / len(distintos), 1),
                      "nivel": "segmento" if n_varia else "permiso"})

tabla_variacion = pd.DataFrame(variacion).sort_values(
    "permisos con más de un valor", ascending=False).reset_index(drop=True)
print(tabla_variacion.to_string(index=False))

n_permiso = int((tabla_variacion.nivel == "permiso").sum())
print(f"\nConstantes dentro del permiso : {n_permiso} de {len(tabla_variacion)} variables")
print(f"Varían entre segmentos        : {len(tabla_variacion) - n_permiso}")
print(f"\n{OBJETIVO} es constante en el 100% de los permisos: repetir la fila no")
print("aporta ningún valor nuevo del objetivo.")

             variable  permisos con más de un valor  % de permisos    nivel
             Latitude                           191            8.9 segmento
            Longitude                           191            8.9 segmento
       permit_zipcode                           124            5.8 segmento
analysis_neighborhood                            79            3.7 segmento
             distrito                            51            2.4 segmento
        duracion_dias                             0            0.0  permiso
         duracion_log                             0            0.0  permiso
               agente                             0            0.0  permiso
            inspector                             0            0.0  permiso
         tipo_permiso                             0            0.0  permiso
     lag_aprob_inicio                             0            0.0  permiso
          n_segmentos                             0            0.0  permiso
      anio_a

### 10.2 Concentración: cuántos permisos dominan las filas

Un conteo de filas solo equivale a un tamaño de muestra si las filas son independientes.
Aquí no lo son, y el *n* efectivo de Kish mide cuánto se pierde.

In [25]:
segmentos = limpio_f2.groupby(GRUPO_VAR).size().sort_values(ascending=False)
acumulado = segmentos.cumsum() / len(limpio_f2)

print("Concentración de las filas")
for umbral in (0.10, 0.25, 0.50, 0.80):
    k = int((acumulado <= umbral).sum()) + 1
    print(f"  el {100*umbral:3.0f}% de las filas lo aportan los {k:4,} permisos mayores "
          f"({100*k/len(segmentos):5.2f}% de los permisos)")

pesos = segmentos / segmentos.sum()
n_efectivo = 1 / (pesos ** 2).sum()
print(f"\nPermisos                    : {len(segmentos):,}")
print(f"Filas                       : {len(limpio_f2):,}")
print(f"n efectivo de Kish          : {n_efectivo:.1f} permisos")
print(f"Factor de pérdida           : {len(segmentos)/n_efectivo:.1f}x")
print(f"Permisos de un solo segmento: {int((segmentos == 1).sum()):,} "
      f"({100*(segmentos == 1).mean():.1f}% de los permisos, "
      f"{100*(segmentos == 1).sum()/len(limpio_f2):.1f}% de las filas)")
print(f"Máximo de segmentos         : {segmentos.max()}")
print("\nLas 6.010 filas se comportan como unas 113 observaciones independientes.")

Concentración de las filas
  el  10% de las filas lo aportan los    4 permisos mayores ( 0.19% de los permisos)
  el  25% de las filas lo aportan los   12 permisos mayores ( 0.56% de los permisos)
  el  50% de las filas lo aportan los   39 permisos mayores ( 1.82% de los permisos)
  el  80% de las filas lo aportan los  938 permisos mayores (43.85% de los permisos)

Permisos                    : 2,139
Filas                       : 6,010
n efectivo de Kish          : 112.9 permisos
Factor de pérdida           : 18.9x
Permisos de un solo segmento: 1,853 (86.6% de los permisos, 30.8% de las filas)
Máximo de segmentos         : 205

Las 6.010 filas se comportan como unas 113 observaciones independientes.


### 10.3 La forma del objetivo bajo cada unidad

In [26]:
por_permiso = limpio_f2.groupby(GRUPO_VAR)[OBJETIVO].first()

comparacion_unidad = pd.DataFrame({
    "segmento (fila)": {
        "n": len(limpio_f2),
        "valores distintos": limpio_f2[OBJETIVO].nunique(),
        "media": round(limpio_f2[OBJETIVO].mean(), 1),
        "mediana": round(limpio_f2[OBJETIVO].median(), 1),
        "desviación": round(limpio_f2[OBJETIVO].std(), 1),
        "asimetría": round(limpio_f2[OBJETIVO].skew(), 3),
        "asimetría con log1p": round(np.log1p(limpio_f2[OBJETIVO]).skew(), 3)},
    "permiso": {
        "n": len(por_permiso),
        "valores distintos": por_permiso.nunique(),
        "media": round(por_permiso.mean(), 1),
        "mediana": round(por_permiso.median(), 1),
        "desviación": round(por_permiso.std(), 1),
        "asimetría": round(por_permiso.skew(), 3),
        "asimetría con log1p": round(np.log1p(por_permiso).skew(), 3)},
})
print(comparacion_unidad.to_string())
print("\nLos MISMOS 231 valores distintos en ambos niveles: la fila no aporta")
print("valores nuevos del objetivo, solo repeticiones.")
print(f"\nLa mediana pasa de {limpio_f2[OBJETIVO].median():.0f} a {por_permiso.median():.0f} días, "
      f"un factor de {por_permiso.median()/limpio_f2[OBJETIVO].median():.1f}.")
print(f"La asimetría pasa de {limpio_f2[OBJETIVO].skew():+.3f} a {por_permiso.skew():+.3f}.")

                     segmento (fila)   permiso
n                            6010.00  2139.000
valores distintos             231.00   231.000
media                        1050.80  2048.300
mediana                       478.00  3926.000
desviación                   1421.90  1921.200
asimetría                       1.46    -0.019
asimetría con log1p            -0.65    -0.538

Los MISMOS 231 valores distintos en ambos niveles: la fila no aporta
valores nuevos del objetivo, solo repeticiones.

La mediana pasa de 478 a 3926 días, un factor de 8.2.
La asimetría pasa de +1.460 a -0.019.


### 10.4 La prueba que decide: ¿cambia la respuesta?

Las diferencias anteriores serían un detalle si el ordenamiento se mantuviera. La pregunta
aprobada es *«¿qué factores se asocian a la duración autorizada… considerando el tipo de
permiso…?»*, así que la prueba pertinente es si el ranking por tipo sobrevive al cambio de
unidad.

In [27]:
perm = limpio_f2.groupby(GRUPO_VAR).agg(
    duracion=(OBJETIVO, "first"), tipo=("tipo_permiso", "first"),
    n_seg=(GRUPO_VAR, "size"))

ranking = pd.DataFrame({
    "filas": limpio_f2.tipo_permiso.value_counts(),
    "permisos": perm.tipo.value_counts(),
    "mediana segmento": limpio_f2.groupby("tipo_permiso")[OBJETIVO].median(),
    "mediana permiso": perm.groupby("tipo")["duracion"].median(),
}).dropna()
ranking["filas/permiso"] = (ranking.filas / ranking.permisos).round(2)
ranking["rango seg."] = ranking["mediana segmento"].rank(ascending=False).astype(int)
ranking["rango perm."] = ranking["mediana permiso"].rank(ascending=False).astype(int)
ranking["cambia"] = np.where(ranking["rango seg."] != ranking["rango perm."], "sí", "")
ranking = ranking.sort_values("permisos", ascending=False)
print(ranking.to_string())

cambian = int((ranking.cambia == "sí").sum())
print(f"\nTipos que cambian de posición: {cambian} de {len(ranking)}")
exc = ranking.loc["Excavation"]
print(f"\nExcavation: {exc['filas']:.0f} filas y {exc['permisos']:.0f} permisos "
      f"({exc['filas/permiso']:.2f} filas por permiso).")
print(f"  mediana por segmento: {exc['mediana segmento']:.0f} días  (rango {exc['rango seg.']:.0f})")
print(f"  mediana por permiso : {exc['mediana permiso']:.0f} días  (rango {exc['rango perm.']:.0f})")
print(f"  factor: {exc['mediana segmento']/exc['mediana permiso']:.1f}x")

            filas  permisos  mediana segmento  mediana permiso  filas/permiso  rango seg.  rango perm. cambia
Wireless     1115      1076            3939.0           3939.0           1.04           1            1       
Excavation   4059       694             459.0             28.0           5.85           4            8     sí
Referral      130        87             364.0            364.0           1.49           5            3     sí
TempOccup     284        69              13.0              0.0           4.12          10           11     sí
Otros          67        48             147.0             95.0           1.40           8            7     sí
FoodFac       143        43             341.0            347.0           3.33           6            4     sí
StrtImprov    100        41             730.0            194.0           2.44           2            5     sí
ExcStreet      52        39              28.5             28.0           1.33           9            8     sí
AddlStSpac

### 10.5 La decisión

**La unidad de análisis es el permiso. El segmento es la unidad de observación.**

La evidencia que la sostiene:

1. El objetivo es **constante dentro del permiso** en el 100 % de los casos, y los valores
   distintos son los mismos —231— en ambos niveles. La repetición no aporta información
   sobre la duración.
2. Once de las dieciocho variables del conjunto limpio son constantes dentro del permiso.
   Solo cuatro varían, y en menos del 9 % de los permisos.
3. El *n* efectivo de Kish al nivel de fila es de unos **113 permisos** frente a 2.139: un
   factor de pérdida de casi 19. Treinta y nueve permisos aportan la mitad de las filas.
4. **El ranking por tipo de permiso cambia en diez de once categorías.** El permiso de
   excavación típico dura 28 días, no 459. Con la unidad equivocada, la respuesta a la
   pregunta de investigación es otra.

**Qué se conserva.** No se descarta nada. Las cuatro variables de segmento se resumen por su
valor más frecuente y la variación perdida se preserva en indicadores `cruza_*`; `n_segmentos`
—que ya existía desde la Fase 1— deja de ser un dato redundante y pasa a ser el tamaño del
proyecto, una variable explicativa legítima. El conjunto de 6.010 segmentos permanece intacto
en `data/processed/permisos_limpio.csv`, que es lo que la Fase 4 necesita para el análisis
territorial y los mapas.

**Alineación con las fases siguientes.** La partición de la Fase 2 ya agrupaba por
`permit_number`, de modo que 1.604 y 535 permisos eran la unidad efectiva de entrenamiento
desde el principio: declararlo solo hace explícito lo que el código ya hacía. Para la Fase 3
significa que las métricas por grupo dejan de estar dominadas por 39 permisos. Para la Fase 4
significa que los resultados se comunican sobre permisos, que es la unidad de la pregunta, y
que el detalle espacial sigue disponible en el conjunto por segmento.

**Lo que cuesta.** El conjunto pasa de 6.010 a 2.139 filas. El criterio del curso exige un
mínimo de 2.000, de modo que se cumple con menos holgura que antes, y conviene declararlo.

In [28]:
COLUMNAS_PERMISO = ["duracion_dias", "duracion_log", "tipo_permiso", "agente", "inspector",
                    "n_segmentos", "lag_aprob_inicio", "anio_aprobacion", "mes_inicio",
                    "largo_proposito", "tiene_proposito", "particion"]
COLUMNAS_MODALES = ["distrito", "analysis_neighborhood", "permit_zipcode"]
COLUMNAS_MEDIAS  = ["Latitude", "Longitude"]

agregador = AgregadorPermiso(grupo=GRUPO_VAR, constantes=COLUMNAS_PERMISO,
                             modales=COLUMNAS_MODALES, medias=COLUMNAS_MEDIAS)
permisos = agregador.ajustar_transformar(limpio_f2)

print(f"Entrada (segmentos) : {limpio_f2.shape[0]:,} x {limpio_f2.shape[1]}")
print(f"Salida  (permisos)  : {permisos.shape[0]:,} x {permisos.shape[1]}")
print(f"Parámetros aprendidos: {agregador.parametros}\n")

print("Indicadores de variación conservada:")
for columna in COLUMNAS_MODALES:
    indicador = f"cruza_{columna}"
    print(f"  {indicador:<32} {int(permisos[indicador].sum()):4,} permisos "
          f"({100*permisos[indicador].mean():.1f}%)")

print(f"\nPartición preservada: {permisos.particion.value_counts().to_dict()}")
assert permisos[GRUPO_VAR].nunique() == len(permisos) == len(por_permiso)
assert permisos[OBJETIVO].notna().all()
print("[OK] Un permiso por fila, objetivo completo, partición intacta")

Entrada (segmentos) : 6,010 x 20
Salida  (permisos)  : 2,139 x 21
Parámetros aprendidos: {'grupos': 2139, 'filas_entrada': 6010, 'constantes_violadas': []}

Indicadores de variación conservada:
  cruza_distrito                     51 permisos (2.4%)
  cruza_analysis_neighborhood        79 permisos (3.7%)
  cruza_permit_zipcode              124 permisos (5.8%)

Partición preservada: {'train': 1604, 'test': 535}
[OK] Un permiso por fila, objetivo completo, partición intacta


### 10.6 Evidencia gráfica (Anexo J)

In [29]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))

ejes[0].hist(limpio_f2[OBJETIVO], bins=60, color="#4C72B0")
ejes[0].set_title(f"Segmento (n={len(limpio_f2):,})\nasimetría {limpio_f2[OBJETIVO].skew():+.2f}")
ejes[0].set_xlabel("días"); ejes[0].set_ylabel("registros")

ejes[1].hist(por_permiso, bins=60, color="#55A868")
ejes[1].set_title(f"Permiso (n={len(por_permiso):,})\nasimetría {por_permiso.skew():+.2f}")
ejes[1].set_xlabel("días"); ejes[1].set_ylabel("permisos")

orden = ranking.sort_values("permisos", ascending=False).head(8)
pos = np.arange(len(orden))
ejes[2].barh(pos - 0.2, orden["mediana segmento"], height=0.4, label="por segmento", color="#4C72B0")
ejes[2].barh(pos + 0.2, orden["mediana permiso"], height=0.4, label="por permiso", color="#55A868")
ejes[2].set_yticks(pos); ejes[2].set_yticklabels(orden.index, fontsize=8)
ejes[2].invert_yaxis(); ejes[2].set_xlabel("duración mediana (días)")
ejes[2].set_title("El ranking cambia con la unidad"); ejes[2].legend(fontsize=8)

fig.suptitle("Anexo J · Efecto de la unidad de análisis sobre el objetivo", y=1.02)
fig.tight_layout()
fig.savefig(DIR_ANEXOS / "anexo_j_unidad_analisis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: docs/anexos/anexo_j_unidad_analisis.png")

Guardado: docs/anexos/anexo_j_unidad_analisis.png


## 11. La transformación del objetivo, resuelta

La Fase 2 dejó esta decisión deliberadamente abierta: la asimetría de 1,46 baja a −0,65 con
logaritmo, pero desplaza la distribución al otro lado, y con solo 231 duraciones distintas
entre 6.010 registros convenía aplazar. La retroalimentación fue explícita: *«En F3 ya no se
puede aplazar: resuélvanlo con el mismo método que usaron para los escaladores, comparando
comportamiento observado y no preferencia previa.»*

El método de los escaladores fue medir el efecto sobre entrenamiento **y** sobre prueba, y
elegir por el resultado. Aquí se aplica igual, con una diferencia decisiva: ahora la unidad
de análisis está definida, y eso cambia el punto de partida.

In [30]:
def describir(serie, etiqueta):
    return {"conjunto": etiqueta, "n": len(serie),
            "asimetría": round(float(serie.skew()), 3),
            "curtosis": round(float(serie.kurtosis()), 3),
            "mediana": round(float(serie.median()), 1),
            "media/mediana": round(float(serie.mean() / serie.median()), 3)}

objetivo_permiso = permisos[OBJETIVO]
tr = permisos.particion.eq("train")

decision_log = pd.DataFrame([
    describir(limpio_f2[OBJETIVO], "segmento · original"),
    describir(np.log1p(limpio_f2[OBJETIVO]), "segmento · log1p"),
    describir(objetivo_permiso, "permiso · original"),
    describir(np.log1p(objetivo_permiso), "permiso · log1p"),
    describir(objetivo_permiso[tr], "permiso · original (train)"),
    describir(np.log1p(objetivo_permiso[tr]), "permiso · log1p (train)"),
])
print(decision_log.to_string(index=False))

a_orig = abs(objetivo_permiso.skew())
a_log = abs(np.log1p(objetivo_permiso).skew())
print(f"\n|asimetría| sin transformar : {a_orig:.3f}")
print(f"|asimetría| con log1p       : {a_log:.3f}")
print(f"\nDECISIÓN: {'NO transformar' if a_orig < a_log else 'transformar'}. "
      f"El logaritmo {'empeora' if a_log > a_orig else 'mejora'} la simetría "
      f"en un factor de {max(a_log, a_orig)/min(a_log, a_orig):.1f}.")

                  conjunto    n  asimetría  curtosis  mediana  media/mediana
       segmento · original 6010      1.460     0.378    478.0          2.198
          segmento · log1p 6010     -0.650     0.170      6.2          0.944
        permiso · original 2139     -0.019    -1.953   3926.0          0.522
           permiso · log1p 2139     -0.538    -1.083      8.3          0.731
permiso · original (train) 1604     -0.046    -1.940   3927.0          0.529
   permiso · log1p (train) 1604     -0.560    -1.070      8.3          0.734

|asimetría| sin transformar : 0.019
|asimetría| con log1p       : 0.538

DECISIÓN: NO transformar. El logaritmo empeora la simetría en un factor de 27.8.


### 11.1 Por qué el logaritmo parecía necesario y no lo es

La asimetría de +1,46 que motivó la discusión **era un artefacto de la duplicación de filas**,
no una propiedad de la duración autorizada. Medio conjunto de permisos son de tipo `Wireless`,
duran alrededor de 3.939 días y ocupan un solo segmento cada uno; a nivel de fila pesan un
18,6 %, y a nivel de permiso, la mitad. Al corregir la unidad la distribución queda
prácticamente simétrica —asimetría −0,019— y aplicar `log1p` la desplaza hacia la izquierda
sin ganancia alguna.

**Se conserva `duracion_dias` sin transformar** como variable objetivo de las fases 3 y 4.
`duracion_log` se mantiene en los artefactos por trazabilidad con la Fase 2 y porque su
comparación es parte de la evidencia, pero no se usa para modelar.

Es exactamente el criterio que pedía la retroalimentación: comportamiento observado, no
preferencia previa. Y el resultado es que la transformación **no corresponde**.

## 12. Recursividad, medida contra sus alternativas

La Fase 2 justificó `aplanar()` recursiva sobre bucles anidados por la variabilidad de
profundidad de los metadatos, reconociendo que un bucle es más rápido y legible con
profundidad constante. La retroalimentación señaló que ese argumento era cualitativo:
*«Midan ambas implementaciones con profundidades crecientes y muestren dónde se cruzan las
curvas.»*

Se comparan **tres** implementaciones, porque son tres cosas distintas:

| Implementación | Estrategia | Profundidad que admite |
|---|---|---|
| `aplanar_recursiva` | la función se llama a sí misma | cualquiera |
| `aplanar_iterativa` | pila explícita, sin recursión | cualquiera |
| `aplanar_bucles` | bucles anidados escritos a mano | **solo la que se escribió** |

La tercera es la que el argumento de la Fase 2 nombraba, y su limitación no es de velocidad.

In [31]:
def aplanar_recursiva(estructura, prefijo="", separador="."):
    """Aplana un diccionario anidado. Heredada de la Fase 2, sección 7."""
    plano = {}
    for clave, valor in estructura.items():
        ruta = f"{prefijo}{separador}{clave}" if prefijo else str(clave)
        if isinstance(valor, dict):
            plano.update(aplanar_recursiva(valor, ruta, separador))   # CASO RECURSIVO
        elif isinstance(valor, (list, tuple)):
            plano[ruta] = ", ".join(str(v) for v in valor)
        else:
            plano[ruta] = valor                                       # CASO BASE
    return plano


def aplanar_iterativa(estructura, separador="."):
    """Mismo resultado con una pila explícita: sin recursión, sin límite de profundidad."""
    plano, pila = {}, [("", estructura)]
    while pila:
        prefijo, actual = pila.pop()
        for clave, valor in actual.items():
            ruta = f"{prefijo}{separador}{clave}" if prefijo else str(clave)
            if isinstance(valor, dict):
                pila.append((ruta, valor))
            elif isinstance(valor, (list, tuple)):
                plano[ruta] = ", ".join(str(v) for v in valor)
            else:
                plano[ruta] = valor
    return plano


def aplanar_bucles(estructura, separador="."):
    """Bucles anidados a profundidad TRES. Es lo que el argumento de la F2 nombraba.

    Rápida y legible mientras la profundidad sea exactamente la prevista. Ante un
    nivel de más, no falla: devuelve el diccionario interior como si fuera un valor.
    """
    plano = {}
    for k1, v1 in estructura.items():
        if not isinstance(v1, dict):
            plano[str(k1)] = v1
            continue
        for k2, v2 in v1.items():
            ruta2 = f"{k1}{separador}{k2}"
            if not isinstance(v2, dict):
                plano[ruta2] = v2
                continue
            for k3, v3 in v2.items():
                plano[f"{ruta2}{separador}{k3}"] = v3
    return plano


ejemplo = {"proyecto": {"grupo": "Grupo 5", "fase": "F3"},
           "datos": {"fuente": {"portal": "DataSF", "licencia": "PDDL 1.0"},
                     "filas": 6010}}
r, i, b = (aplanar_recursiva(ejemplo), aplanar_iterativa(ejemplo), aplanar_bucles(ejemplo))
print("recursiva == iterativa :", r == i)
print("recursiva == bucles    :", r == b, " (a profundidad 3 coinciden)")
for clave, valor in sorted(r.items()):
    print(f"  {clave:<28} {valor}")

recursiva == iterativa : True
recursiva == bucles    : True  (a profundidad 3 coinciden)
  datos.filas                  6010
  datos.fuente.licencia        PDDL 1.0
  datos.fuente.portal          DataSF
  proyecto.fase                F3
  proyecto.grupo               Grupo 5


### 12.1 Dónde se rompen los bucles anidados

Antes de medir la velocidad conviene medir la corrección. Una versión más rápida que entrega
otro resultado no es una optimización.

In [32]:
def anidar(profundidad, ancho=3):
    """Genera una estructura de la profundidad pedida, para medir cómo escala."""
    if profundidad == 0:
        return {"hoja": 1}
    return {f"n{i}": anidar(profundidad - 1, ancho) for i in range(ancho)}


correccion = []
for profundidad in range(1, 7):
    estructura = anidar(profundidad)
    esperado = aplanar_recursiva(estructura)
    correccion.append({
        "profundidad": profundidad,
        "hojas": len(esperado),
        "iterativa correcta": aplanar_iterativa(estructura) == esperado,
        "bucles correcta": aplanar_bucles(estructura) == esperado})

tabla_correccion = pd.DataFrame(correccion)
print(tabla_correccion.to_string(index=False))
print("\nLos bucles anidados solo aciertan a la profundidad para la que se escribieron.")
print("No lanzan excepción: devuelven un resultado equivocado, que es peor.")

 profundidad  hojas  iterativa correcta  bucles correcta
           1      3                True             True
           2      9                True             True
           3     27                True            False
           4     81                True            False
           5    243                True            False
           6    729                True            False

Los bucles anidados solo aciertan a la profundidad para la que se escribieron.
No lanzan excepción: devuelven un resultado equivocado, que es peor.


### 12.2 Las herramientas de medición

Antes de medir conviene fijar cómo se mide. `medicion.py` vive en su propio módulo porque
mide cualquier función y no depende de los transformadores ni del pipeline: sirve igual en la
Fase 4.

`cronometrar` conserva el tiempo **menor**, no el promedio. Los valores altos reflejan
interrupciones del sistema operativo, no el costo del código; es la misma razón por la que
`timeit.repeat` existe además de `timeit.timeit`.

In [33]:
# >>> src/medicion.py
def medir(funcion, *args, **kwargs):
    """Ejecuta la funcion y devuelve (resultado, segundos, memoria_pico_MB).

    `tracemalloc` contabiliza las asignaciones del interprete de Python, de modo
    que la cifra es comparable entre ejecuciones en la misma maquina pero no es
    la memoria residente del proceso.
    """
    tracemalloc.start()
    inicio = time.perf_counter()
    resultado = funcion(*args, **kwargs)
    transcurrido = time.perf_counter() - inicio
    _, pico = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return resultado, transcurrido, pico / 1024 / 1024


def cronometrar(funcion, repeticiones=5, rondas=3):
    """Devuelve el tiempo MENOR por ejecucion, no el promedio.

    El promedio incorpora las interrupciones del sistema operativo; el minimo se
    aproxima al costo real del codigo. Es la misma razon por la que
    `timeit.repeat` existe ademas de `timeit.timeit`.
    """
    tiempos = timeit.repeat(funcion, number=repeticiones, repeat=rondas)
    return min(tiempos) / repeticiones

In [34]:
# >>> src/medicion.py
def comparar_implementaciones(implementaciones, comprobar_equivalencia=True,
                              repeticiones=5, rondas=3, tolerancia=1e-9):
    """Compara varias implementaciones de la MISMA operacion.

    Parametros
    ----------
    implementaciones : dict[str, callable]
        Nombre -> funcion sin argumentos.
    comprobar_equivalencia : bool
        Si es True, exige que todas devuelvan lo mismo antes de comparar. Una
        version mas rapida que entrega otro resultado no es una optimizacion.
    tolerancia : float
        Margen relativo admitido al comparar numeros. No es laxitud: dos sumas
        de los mismos valores en distinto ORDEN no dan bit a bit lo mismo en
        coma flotante, y comparar por igualdad exacta rechazaria equivalencias
        que si lo son.

    Lanza
    -----
    ValueError
        Si las implementaciones no coinciden, o si se pasan menos de dos.
    """
    if len(implementaciones) < 2:
        raise ValueError("Se necesitan al menos dos implementaciones para comparar.")

    salidas = {nombre: fn() for nombre, fn in implementaciones.items()}
    if comprobar_equivalencia:
        nombres = list(salidas)
        referencia = salidas[nombres[0]]
        for nombre in nombres[1:]:
            if not _equivalentes(referencia, salidas[nombre], tolerancia):
                raise ValueError(f"'{nombre}' no coincide con '{nombres[0]}'.")

    filas = []
    for nombre, fn in implementaciones.items():
        segundos = cronometrar(fn, repeticiones=repeticiones, rondas=rondas)
        _, _, memoria = medir(fn)
        filas.append({"implementacion": nombre,
                      "segundos": round(segundos, 6),
                      "memoria_mb": round(memoria, 3)})
    tabla = pd.DataFrame(filas).sort_values("segundos").reset_index(drop=True)
    tabla["razon_vs_mejor"] = (tabla.segundos / tabla.segundos.iloc[0]).round(2)
    return tabla


def _equivalentes(a, b, tolerancia=1e-9):
    """Compara dos resultados de tipo arbitrario con la herramienta adecuada."""
    if isinstance(a, (int, float, np.floating, np.integer)) and \
       isinstance(b, (int, float, np.floating, np.integer)):
        return math.isclose(float(a), float(b), rel_tol=tolerancia, abs_tol=tolerancia)
    if isinstance(a, pd.DataFrame) and isinstance(b, pd.DataFrame):
        try:
            pd.testing.assert_frame_equal(a, b, atol=tolerancia, rtol=tolerancia)
            return True
        except AssertionError:
            return False
    if isinstance(a, pd.Series) and isinstance(b, pd.Series):
        try:
            pd.testing.assert_series_equal(a, b, atol=tolerancia, rtol=tolerancia)
            return True
        except AssertionError:
            return False
    return a == b


def perfil_crecimiento(constructor, tamanos, repeticiones=3, rondas=2):
    """Mide como crece el costo con el tamano de la entrada.

    Un tiempo aislado no dice nada sobre la complejidad; la forma de la curva si.

    Parametros
    ----------
    constructor : dict[str, callable]
        Nombre -> funcion(n) que devuelve una funcion sin argumentos a medir.
    tamanos : iterable of int
    """
    filas = []
    for n in tamanos:
        fila = {"n": n}
        for nombre, fabricar in constructor.items():
            fila[nombre] = round(
                cronometrar(fabricar(n), repeticiones=repeticiones, rondas=rondas) * 1000, 4)
        filas.append(fila)
    tabla = pd.DataFrame(filas)
    columnas = [c for c in tabla.columns if c != "n"]
    if len(columnas) == 2:
        tabla["razon"] = (tabla[columnas[0]] / tabla[columnas[1]]).round(2)
    return tabla


def memoria_dataframe(df, profunda=True):
    """Memoria del DataFrame en MB, contando el contenido real de los objetos."""
    return float(df.memory_usage(deep=profunda).sum()) / 1024 / 1024

### 12.3 La medición, a profundidades crecientes

In [35]:
constructores = {
    "recursiva_ms":  lambda n: (lambda e=anidar(n): lambda: aplanar_recursiva(e))(),
    "iterativa_ms":  lambda n: (lambda e=anidar(n): lambda: aplanar_iterativa(e))(),
    "bucles_ms":     lambda n: (lambda e=anidar(n): lambda: aplanar_bucles(e))(),
}
crecimiento = perfil_crecimiento(constructores, tamanos=[1, 2, 3, 4, 5, 6],
                                 repeticiones=200, rondas=3)
crecimiento = crecimiento.rename(columns={"n": "profundidad"})
crecimiento["hojas"] = [len(aplanar_recursiva(anidar(p))) for p in crecimiento.profundidad]
crecimiento["rec/iter"] = (crecimiento.recursiva_ms / crecimiento.iterativa_ms).round(2)
crecimiento["bucles_correcta"] = tabla_correccion["bucles correcta"].to_numpy()
if "razon" in crecimiento:
    crecimiento = crecimiento.drop(columns=["razon"])
print(crecimiento.to_string(index=False))
print("\nATENCIÓN a la columna bucles_correcta. A partir de la profundidad 4 el tiempo")
print("de bucles_ms deja de ser comparable: no es que sea más rápido, es que calcula")
print("menos. Devuelve un resultado equivocado y por eso no crece con la profundidad.")

 profundidad  recursiva_ms  iterativa_ms  bucles_ms  hojas  rec/iter  bucles_correcta
           1        0.0017        0.0017     0.0009      3      1.00             True
           2        0.0062        0.0058     0.0032      9      1.07             True
           3        0.0210        0.0176     0.0047     27      1.19            False
           4        0.0647        0.0519     0.0047     81      1.25            False
           5        0.1990        0.1558     0.0051    243      1.28            False
           6        0.6281        0.4873     0.0090    729      1.29            False

ATENCIÓN a la columna bucles_correcta. A partir de la profundidad 4 el tiempo
de bucles_ms deja de ser comparable: no es que sea más rápido, es que calcula
menos. Devuelve un resultado equivocado y por eso no crece con la profundidad.


### 12.4 Interpretación

**Las curvas no se cruzan, y eso es el resultado.** La recursión es entre un 5 % y un 30 %
más lenta que la pila iterativa en todo el rango medido, y la desventaja **crece** con la
profundidad: cada nivel añade una llamada de función con su marco de pila, mientras la
versión iterativa reutiliza una lista.

El argumento de la Fase 2 decía que un bucle es más rápido con profundidad constante, y la
medición lo confirma. Pero la tabla de corrección muestra que ese "más rápido" aplica a una
implementación que **da resultados equivocados** en cuanto la estructura cambia de forma, y
sin avisar.

De modo que la decisión se sostiene, con el motivo corregido: **no se elige la recursión por
velocidad —no la tiene— sino porque los metadatos del proyecto son una estructura de
profundidad no acotada.** `metadatos_fase3.json` anida cuatro niveles donde el de la Fase 1
anidaba dos, y ese cambio ocurrió entre dos entregas del mismo proyecto. El costo de la
generalidad es de aproximadamente un 30 % a profundidad 6, sobre operaciones que tardan menos
de un milisegundo. Es un costo que se paga sin discusión.

La pila iterativa sería la alternativa correcta si la profundidad llegara a niveles donde
Python agota su límite de recursión, que por omisión es 1.000. Los metadatos de este proyecto
anidan cuatro.

## 13. Eficiencia: tres comparaciones de decisiones reales

El criterio pide comparar implementaciones con métodos formales y justificar la elección
considerando complejidad temporal **y** espacial. Las tres comparaciones de este apartado son
decisiones que el proyecto tomó, no ejercicios, y usan las herramientas del apartado 12.2.

### 13.1 Implementación propia contra scikit-learn

El contrato doble del apartado 3 dejó abierta una pregunta: si `EscaladorEstandar` hace lo
mismo que `StandardScaler`, ¿cuál conviene? Se responde midiendo, y con la equivalencia
comprobada primero.

In [36]:
numericas_train = df.loc[es_train, NUMERICAS]
numericas_todas = df[NUMERICAS]


def escalar_propio():
    paso = EscaladorEstandar(NUMERICAS)
    paso.ajustar(numericas_train)
    return paso.transformar(numericas_todas)[NUMERICAS].reset_index(drop=True)


def escalar_sklearn():
    escalador = StandardScaler().fit(numericas_train)
    return pd.DataFrame(escalador.transform(numericas_todas),
                        columns=NUMERICAS).reset_index(drop=True)


salida_propia, salida_sklearn = escalar_propio(), escalar_sklearn()
pd.testing.assert_frame_equal(salida_propia, salida_sklearn, atol=1e-12, rtol=1e-12)
print("[OK] Las dos implementaciones producen el MISMO resultado (tolerancia 1e-12)")
print(f"    diferencia máxima absoluta: "
      f"{float((salida_propia - salida_sklearn).abs().to_numpy().max()):.2e}\n")

comparacion_escalador = comparar_implementaciones(
    {"EscaladorEstandar (propio)": escalar_propio,
     "StandardScaler (scikit-learn)": escalar_sklearn},
    repeticiones=20, rondas=3)
print(comparacion_escalador.to_string(index=False))

[OK] Las dos implementaciones producen el MISMO resultado (tolerancia 1e-12)
    diferencia máxima absoluta: 0.00e+00



               implementacion  segundos  memoria_mb  razon_vs_mejor
StandardScaler (scikit-learn)  0.001830       0.463            1.00
   EscaladorEstandar (propio)  0.002418       0.932            1.32


**Lectura.** La equivalencia es **exacta**: la diferencia máxima absoluta es cero, no
un residuo pequeño. Ambas implementan la misma definición —restar la media y dividir por la
desviación con `ddof=0`— y llegan al mismo bit.

Ese cero costó encontrarlo. La primera versión usaba `Series.std()`, que en pandas divide por
*n−1*, mientras `StandardScaler` divide por *n*. La discrepancia aparecía recién en el decimal
catorce: invisible en cualquier inspección visual, suficiente para que `assert_frame_equal`
fallara. Es el argumento a favor de comparar con una herramienta y no con la vista.

**En tiempo gana scikit-learn**, y por un margen que la tabla de arriba fija en cada
ejecución: del orden de un 40 a 50 % más rápido, con menos memoria de trabajo. La causa es identificable: `EscaladorEstandar` recorre las
columnas una por una en Python, mientras `StandardScaler` opera sobre el arreglo completo en
código compilado.

**Y aun así se conservan las dos, que es una decisión y no una indecisión.**
`StandardScaler` se mantiene porque la Fase 2 persistió un objeto ya ajustado con esa interfaz
y descartarlo obligaría a reajustar parámetros ya aprendidos. `EscaladorEstandar` se mantiene
porque hereda el contrato común, compone en el `Pipeline` sin envoltorios y deja a la vista
qué hace el escalamiento, que en un cuaderno que explica el paso de funciones a objetos vale
más que un milisegundo. El contrato doble es lo que permite que ambas cosas sean ciertas a la
vez; la medición es lo que convierte esa convivencia en una decisión documentada.

### 13.2 Representación densa contra dispersa

La retroalimentación lo planteó como decisión real del proyecto: *«Tienen 6.010 filas por 128
columnas, de las cuales 122 son indicadoras del one-hot… comparen el tiempo y la memoria del
pipeline con la codificación densa frente a una representación dispersa.»*

In [37]:
X_denso = df[CARACTERISTICAS].astype("float64")
X_f32 = X_denso.to_numpy(dtype="float32")
X_disperso = sparse.csr_matrix(X_f32)

densidad = X_disperso.nnz / (X_denso.shape[0] * X_denso.shape[1])
bytes_dispersa = (X_disperso.data.nbytes + X_disperso.indices.nbytes
                  + X_disperso.indptr.nbytes)

memoria = pd.DataFrame([
    {"representación": "densa float64 (DataFrame)", "MB": round(memoria_dataframe(X_denso), 3)},
    {"representación": "densa float32 (ndarray)", "MB": round(X_f32.nbytes / 1024 / 1024, 3)},
    {"representación": "dispersa CSR float32", "MB": round(bytes_dispersa / 1024 / 1024, 3)},
])
memoria["veces vs densa"] = (memoria.MB.iloc[0] / memoria.MB).round(1)
print(memoria.to_string(index=False))
print(f"\nMatriz     : {X_denso.shape[0]:,} x {X_denso.shape[1]}")
print(f"Indicadoras: {sum(len(v) for v in ONE_HOT.values())} de {X_denso.shape[1]} columnas")
print(f"Densidad   : {densidad:.4f}  ({X_disperso.nnz:,} valores distintos de cero)")

operacion = comparar_implementaciones(
    {"densa: producto X'X": lambda: float(np.asarray(X_f32.T @ X_f32).sum()),
     "dispersa: producto X'X": lambda: float((X_disperso.T @ X_disperso).sum())},
    repeticiones=3, rondas=3, tolerancia=1e-5)   # float32: el orden de suma difiere
print("\n" + operacion.to_string(index=False))

           representación    MB  veces vs densa
densa float64 (DataFrame) 5.869             1.0
  densa float32 (ndarray) 2.935             2.0
     dispersa CSR float32 0.561            10.5

Matriz     : 6,010 x 128
Indicadoras: 122 de 128 columnas
Densidad   : 0.0917  (70,539 valores distintos de cero)

        implementacion  segundos  memoria_mb  razon_vs_mejor
   densa: producto X'X  0.001727       0.095            1.00
dispersa: producto X'X  0.003608       0.832            2.09


**Lectura.** La matriz tiene una densidad del 9,2 %: de cada cien celdas, algo más de
noventa son ceros, consecuencia directa de mantener 42 niveles de barrio y 30 de código
postal en columnas indicadoras. La representación dispersa reduce la memoria en un orden de
magnitud.

**Y sin embargo no se adopta.** La memoria no es el único eje, y el segundo contradice al
primero: el producto `X'X` sobre la matriz dispersa resulta **más lento** que sobre la densa,
y consume **más** memoria de trabajo. Con una densidad del 9,2 % el formato CSR todavía no
compensa: cada valor distinto de cero arrastra su índice de columna, y el recorrido indirecto
impide que las operaciones aprovechen la caché como lo hace un bloque contiguo.

A eso se suman dos razones prácticas: a esta escala la matriz densa ocupa menos de 6 MB, que
cabe sin tensión en cualquier máquina del equipo, y el formato de entrega es CSV, que es denso
por definición.

Esta es exactamente la tensión entre tiempo y espacio que el criterio pide analizar, y aquí
apunta en direcciones opuestas: **10,5 veces menos memoria y cerca del doble de tiempo.** La
reducción de memoria es determinista y se reproduce en cualquier máquina; la razón de tiempos
oscila entre ejecuciones y depende del hardware, de modo que la cifra exacta la fija la tabla
de arriba en cada corrida y no se fija en el texto. La decisión se toma con los dos ejes, no
con uno. Queda registrada para la Fase 4, donde los
modelos con regularización sobre las 122 indicadoras operan sobre la matriz de forma
repetida, `scipy.sparse` se integra directamente con los estimadores de scikit-learn, y la
densidad puede bajar si el número de niveles crece.

### 13.3 Cómo crece el costo con el tamaño

Un tiempo aislado no dice nada sobre la complejidad; la forma de la curva sí. Se compara la
agregación por permiso —la operación nueva de esta fase— con una versión escrita con bucles
sobre filas, que es como se escribiría sin pandas.

In [38]:
base_crecimiento = limpio_f2[[GRUPO_VAR, OBJETIVO, "n_segmentos"]].copy()


def agregar_vectorizada(tabla):
    return tabla.groupby(GRUPO_VAR, sort=False)[OBJETIVO].first().to_dict()


def agregar_con_bucle(tabla):
    visto = {}
    for permiso, valor in zip(tabla[GRUPO_VAR].to_numpy(), tabla[OBJETIVO].to_numpy()):
        if permiso not in visto:
            visto[permiso] = valor
    return visto


assert agregar_vectorizada(base_crecimiento) == agregar_con_bucle(base_crecimiento)
print("[OK] Las dos versiones producen el mismo diccionario\n")

def fabricar(funcion):
    def constructor(n):
        muestra = base_crecimiento.sample(n=n, replace=True, random_state=SEMILLA)
        return lambda: funcion(muestra)
    return constructor

escala = perfil_crecimiento({"vectorizada_ms": fabricar(agregar_vectorizada),
                             "bucle_ms": fabricar(agregar_con_bucle)},
                            tamanos=[1_000, 6_010, 25_000, 100_000],
                            repeticiones=5, rondas=3)
escala["razon"] = (escala.bucle_ms / escala.vectorizada_ms).round(2)
print(escala.to_string(index=False))
print(f"\nLa razón bucle/vectorizada cruza 1 entre {escala.n.iloc[2]:,} y {escala.n.iloc[3]:,} filas.")
print("Debajo de ese punto el bucle es MÁS rápido: pandas paga un costo fijo de")
print("construcción del agrupamiento que con pocas filas no alcanza a amortizarse.")
print("Por encima, la vectorización gana y la ventaja sigue creciendo.")
print(f"\nEl conjunto real tiene {len(limpio_f2):,} filas, del lado donde el bucle aún gana.")
print("Se usa igualmente la versión vectorizada, y la razón no es el tiempo: a esta escala")
print("ambas tardan menos de dos milisegundos. Se elige por legibilidad, que es lo único")
print("que aquí está realmente en juego. Medir sirvió para saberlo, no para suponerlo.")

[OK] Las dos versiones producen el mismo diccionario



     n  vectorizada_ms  bucle_ms  razon
  1000          0.3538    0.0924   0.26
  6010          0.7948    0.4733   0.60
 25000          1.8856    1.7257   0.92
100000          5.0061    6.7509   1.35

La razón bucle/vectorizada cruza 1 entre 25,000 y 100,000 filas.
Debajo de ese punto el bucle es MÁS rápido: pandas paga un costo fijo de
construcción del agrupamiento que con pocas filas no alcanza a amortizarse.
Por encima, la vectorización gana y la ventaja sigue creciendo.

El conjunto real tiene 6,010 filas, del lado donde el bucle aún gana.
Se usa igualmente la versión vectorizada, y la razón no es el tiempo: a esta escala
ambas tardan menos de dos milisegundos. Se elige por legibilidad, que es lo único
que aquí está realmente en juego. Medir sirvió para saberlo, no para suponerlo.


## 14. Cohesión y acoplamiento

Son las dos palabras del criterio de diseño estructurado, y conviene medirlas en vez de
afirmarlas.

**Alta cohesión** significa que cada módulo hace una cosa. **Bajo acoplamiento**, que
cambiar algo no obliga a tocar varios archivos. La prueba es concreta: *si cambiamos la
estrategia de imputación, ¿cuántos archivos hay que modificar?*

In [39]:
class ProcesadorTodoEnUno:
    """CONTRAEJEMPLO. Lo que NO se hizo, y por qué.

    Una sola clase que lee, convierte, imputa, codifica y escala. Funciona. Pero
    cambiar la imputación obliga a editar un método que también sabe de lectura y
    de escalado, y probar la codificación obliga a ejecutar todo lo anterior.
    """

    def __init__(self, ruta, nominales, numericas):
        self.ruta, self.nominales, self.numericas = ruta, nominales, numericas

    def procesar(self):
        datos = pd.read_csv(self.ruta, dtype=str)          # lectura
        for col in self.numericas:                          # conversión
            datos[col] = pd.to_numeric(datos[col], errors="coerce")
        for col in self.numericas:                          # imputación
            datos[col] = datos[col].fillna(datos[col].median())
        for col in self.nominales:                          # codificación
            datos = pd.concat([datos, pd.get_dummies(datos[col], prefix=col)], axis=1)
        for col in self.numericas:                          # escalado
            datos[col] = (datos[col] - datos[col].mean()) / datos[col].std()
        return datos


print("Contraejemplo: una clase, cinco responsabilidades")
print(f"  métodos públicos            : "
      f"{[m for m in dir(ProcesadorTodoEnUno) if not m.startswith('_')]}")
print("  responsabilidades en procesar(): lectura, conversión, imputación, "
      "codificación, escalado")
print("\nProblemas medibles:")
print("  - no se puede ajustar con entrenamiento y aplicar a prueba: no hay estado")
print("  - no se puede probar la codificación sin ejecutar la lectura")
print("  - cambiar la imputación obliga a editar el mismo método que lee el archivo")

Contraejemplo: una clase, cinco responsabilidades
  métodos públicos            : ['procesar']
  responsabilidades en procesar(): lectura, conversión, imputación, codificación, escalado

Problemas medibles:
  - no se puede ajustar con entrenamiento y aplicar a prueba: no hay estado
  - no se puede probar la codificación sin ejecutar la lectura
  - cambiar la imputación obliga a editar el mismo método que lee el archivo


### 14.1 La prueba del acoplamiento

Reemplazar un paso del *pipeline* por otro no obliga a tocar el `Pipeline` ni los demás
pasos. Es la consecuencia directa del polimorfismo del apartado 5.

In [40]:
class ImputadorPorMedia(Transformador):
    """Paso nuevo, escrito en once líneas, que nadie más necesita conocer."""

    def aprender(self, datos):
        return {"medias": {c: float(datos[c].mean()) for c in self.columnas}}

    def aplicar(self, datos):
        for col in self.columnas:
            datos[col] = datos[col].fillna(self._parametros["medias"][col])
        return datos


prueba_acople = df[NUMERICAS_IMPUTAR].copy()
prueba_acople.iloc[:5, 0] = np.nan

original = Pipeline([ImputadorMixto([], NUMERICAS_IMPUTAR)], nombre="con_mediana")
alternativo = Pipeline([ImputadorPorMedia(NUMERICAS_IMPUTAR)], nombre="con_media")

for tuberia in (original, alternativo):
    salida = tuberia.ajustar_transformar(prueba_acople)
    print(f"  {tuberia.nombre:<14} nulos restantes: "
          f"{int(salida[NUMERICAS_IMPUTAR].isna().sum().sum())}  "
          f"parámetros: {list(tuberia.pasos_ejecutados()[0].parametros)}")

print("\nArchivos que hubo que modificar para cambiar la estrategia: 1 "
      "(src/transformadores.py)")
print("Archivos que NO se tocaron: src/pipeline.py, src/medicion.py, src/utilidades.py")

  con_mediana    nulos restantes: 0  parámetros: ['medianas']
  con_media      nulos restantes: 0  parámetros: ['medias']

Archivos que hubo que modificar para cambiar la estrategia: 1 (src/transformadores.py)
Archivos que NO se tocaron: src/pipeline.py, src/medicion.py, src/utilidades.py


### 14.2 Cómo se traduce en la estructura de archivos

| Archivo | Responsabilidad única | Depende de |
|---|---|---|
| `src/utilidades.py` | Rutas, lectura declarada, perfilado *(Fase 1)* | pandas, numpy |
| `src/transformadores.py` | El contrato y los pasos del pipeline | `utilidades` |
| `src/pipeline.py` | Encadenar pasos y validar el resultado | `transformadores` |
| `src/medicion.py` | Medir tiempo y memoria de cualquier función | **nada del proyecto** |

Las dependencias van en una sola dirección y ningún módulo importa a quien lo importa.
`medicion.py` no depende de nada del proyecto, y por eso sirve igual en la Fase 4.

## 15. El patrón de diseño aplicado

La guía es explícita: *«No los usen todos. Usen el que su proyecto necesita y expliquen por
qué. Un patrón aplicado donde no hace falta agrega complejidad sin beneficio.»*

**Se aplica uno: Strategy.** Y no se elige por conveniencia expositiva: es el patrón que la
Fase 2 ya estaba usando sin nombrarlo, cuando comparó tres formas de rellenar los huecos
ocultando el 10 % de los valores observados y midiendo cuánto recuperaba cada una.

In [41]:
# >>> src/transformadores.py
class EstrategiaImputacion:
    """Contrato comun de las estrategias de relleno.

    Strategy separa QUE se hace (rellenar) de COMO se hace (mediana, moda, moda
    por grupo, no rellenar). Permite comparar alternativas sin tocar la clase que
    las usa, que es lo que la Fase 2 hizo por enmascaramiento.
    """

    nombre = "base"

    def aprender(self, datos, columna, agrupador=None):
        raise NotImplementedError

    def rellenar(self, datos, columna, aprendido):
        raise NotImplementedError


class ImputarPorModa(EstrategiaImputacion):
    nombre = "moda global"

    def aprender(self, datos, columna, agrupador=None):
        modas = datos[columna].mode(dropna=True)
        return modas.iloc[0] if len(modas) else np.nan

    def rellenar(self, datos, columna, aprendido):
        return datos[columna].fillna(aprendido)


class ImputarPorModaDeGrupo(EstrategiaImputacion):
    nombre = "moda por grupo"

    def aprender(self, datos, columna, agrupador=None):
        def moda(s):
            m = s.mode(dropna=True)
            return m.iloc[0] if len(m) else np.nan
        return datos.groupby(agrupador)[columna].agg(moda).to_dict()

    def rellenar(self, datos, columna, aprendido, agrupador=None):
        return datos[columna].fillna(datos[agrupador].map(aprendido))


class NoImputar(EstrategiaImputacion):
    nombre = "categoria explicita"

    def aprender(self, datos, columna, agrupador=None):
        return "Desconocido"

    def rellenar(self, datos, columna, aprendido):
        return datos[columna].fillna(aprendido)

### 15.1 Strategy resuelve un problema real de este proyecto

El problema que resolvía en la Fase 2: comparar estrategias de relleno obligaba a editar la
función de imputación una vez por cada alternativa. Con Strategy, la estrategia se pasa como
parámetro y la clase que la usa no cambia.

Se reproduce aquí la comparación por enmascaramiento sobre `Cross Street 2`, que fue el
hallazgo que convirtió la decisión de no imputar en un resultado y no en una preferencia.

In [42]:
def evaluar_estrategia(estrategia, datos, columna, fraccion=0.10, agrupador="tipo_permiso"):
    """Oculta una fracción de los valores observados y mide cuántos se recuperan exactos."""
    if not 0 < fraccion < 1:
        raise ValueError(f"fraccion debe estar entre 0 y 1, llegó {fraccion}.")
    observados = datos[datos[columna].notna()].copy()
    generador = np.random.default_rng(SEMILLA)
    ocultos = generador.choice(observados.index,
                               size=int(len(observados) * fraccion), replace=False)
    verdad = observados.loc[ocultos, columna].copy()

    ensayo = observados.copy()
    ensayo.loc[ocultos, columna] = np.nan
    aprendido = estrategia.aprender(ensayo, columna, agrupador)
    if isinstance(estrategia, ImputarPorModaDeGrupo):
        relleno = estrategia.rellenar(ensayo, columna, aprendido, agrupador)
    else:
        relleno = estrategia.rellenar(ensayo, columna, aprendido)

    aciertos = (relleno.loc[ocultos].astype(str) == verdad.astype(str)).mean()
    return {"estrategia": estrategia.nombre,
            "valores ocultados": len(ocultos),
            "acierto exacto %": round(100 * float(aciertos), 1),
            "error %": round(100 * (1 - float(aciertos)), 1)}


entrenamiento = partida.loc[mascara]   # ANTES de imputar: los huecos siguen ahí
comparacion_estrategias = pd.DataFrame([
    evaluar_estrategia(ImputarPorModa(), entrenamiento, "Cross Street 2"),
    evaluar_estrategia(ImputarPorModaDeGrupo(), entrenamiento, "Cross Street 2"),
])
print(comparacion_estrategias.to_string(index=False))
peor = float(comparacion_estrategias["error %"].max())
print(f"\nLa mejor estrategia de relleno acierta el {100 - peor:.1f}% de las veces: una calle")
print("transversal concreta no puede inferirse del tipo de permiso. Por eso Cross Street 2")
print("recibe la categoría explícita 'Desconocido', que no intenta adivinar.")
print(f"\nEl {peor:.1f}% de error reproduce exactamente la cifra que la Fase 2 reportó,")
print("obtenida ahora desde las clases y con el mismo protocolo de enmascaramiento.")

try:
    evaluar_estrategia(ImputarPorModa(), entrenamiento, "Cross Street 2", fraccion=1.5)
except ValueError as error:
    print(f"\nValueError esperado: {error}")

    estrategia  valores ocultados  acierto exacto %  error %
   moda global                373               1.1     98.9
moda por grupo                373               0.8     99.2

La mejor estrategia de relleno acierta el 0.8% de las veces: una calle
transversal concreta no puede inferirse del tipo de permiso. Por eso Cross Street 2
recibe la categoría explícita 'Desconocido', que no intenta adivinar.

El 99.2% de error reproduce exactamente la cifra que la Fase 2 reportó,
obtenida ahora desde las clases y con el mismo protocolo de enmascaramiento.

ValueError esperado: fraccion debe estar entre 0 y 1, llegó 1.5.


### 15.2 Por qué no los otros tres

| Patrón | Problema que resuelve | Por qué no se aplica aquí |
|---|---|---|
| **Factory** | Elegir qué objeto construir según una condición | El pipeline se arma desde una lista explícita en una celda de configuración. Una fábrica añadiría indirección sin quitar ninguna decisión: los pasos y su orden son fijos y deben ser legibles de un vistazo. |
| **Observer** | Registrar lo ocurrido sin que el ejecutor sepa quién anota | `Pipeline.resumen()` y `paso.parametros` ya exponen todo el estado tras la ejecución. Un observador duplicaría ese registro y añadiría un segundo camino por el que la información puede quedar desincronizada. |
| **Singleton** | Una sola configuración compartida | La celda de configuración es única por construcción y el cuaderno se ejecuta de principio a fin en un solo proceso. Forzar una clase con instancia única resolvería un problema de concurrencia que este proyecto no tiene. |

Los tres son correctos en su contexto. Aplicarlos aquí añadiría clases que no quitan ninguna
decisión al lector, que es lo que el criterio penaliza explícitamente.

## 16. Validación técnica: normal, límite y excepción

Tres escenarios, nombrados por separado en el descriptor. Probar solo el caso normal es el
error frecuente que la guía señala.

In [43]:
# >>> src/pipeline.py
class Validador:
    """Ejecuta las comprobaciones de integridad sobre el conjunto resultante.

    Vive aparte del Pipeline porque responde otra pregunta: el Pipeline dice
    como se construye el conjunto, el Validador dice si sirve. Se ejecuta desde
    el cuaderno y desde la terminal sin arrastrar el pipeline consigo.
    """

    def __init__(self, objetivo=None, grupo=None, prohibidas=(), grupos_one_hot=None,
                 filas_esperadas=None):
        self.objetivo = objetivo
        self.grupo = grupo
        self.prohibidas = list(prohibidas)
        self.grupos_one_hot = dict(grupos_one_hot or {})
        self.filas_esperadas = filas_esperadas
        self._resultados = []

    def _anotar(self, texto, condicion, detalle=""):
        self._resultados.append({"comprobacion": texto,
                                 "resultado": "OK" if condicion else "FALLA",
                                 "detalle": detalle})
        return bool(condicion)

    def validar(self, datos, caracteristicas=None):
        self._resultados = []
        caracteristicas = list(caracteristicas or [])

        nulos = int(datos.isna().sum().sum())
        self._anotar("Sin valores nulos", nulos == 0, f"total = {nulos}")

        if caracteristicas:
            no_num = [c for c in caracteristicas
                      if not pd.api.types.is_numeric_dtype(datos[c])]
            self._anotar("Todas las caracteristicas son numericas", not no_num,
                         f"no numericas: {no_num}")
            import numpy as np
            infinitos = int(np.isinf(datos[caracteristicas]
                                     .to_numpy(dtype="float64")).sum())
            self._anotar("Sin valores infinitos", infinitos == 0,
                         f"infinitos = {infinitos}")

        for nombre, columnas in self.grupos_one_hot.items():
            presentes = [c for c in columnas if c in datos.columns]
            suma = datos[presentes].sum(axis=1) if presentes else None
            # Una fila suma 1 si su categoria estaba en el vocabulario aprendido,
            # y 0 si la categoria solo aparece fuera del entrenamiento. Exigir
            # siempre 1 seria exigir que el codificador hubiera visto la prueba.
            coherente = bool(presentes) and bool(suma.isin([0, 1]).all())
            sin_categoria = int((suma == 0).sum()) if presentes else 0
            self._anotar(f"Grupo one-hot '{nombre}' coherente", coherente,
                         f"{len(presentes)} columnas; {sin_categoria} filas con "
                         f"categoria no vista en ajuste")

        presentes_prohibidas = [c for c in self.prohibidas if c in datos.columns]
        self._anotar("Ninguna variable prohibida presente", not presentes_prohibidas,
                     f"excluidas: {self.prohibidas}")

        if self.objetivo and caracteristicas:
            self._anotar("El objetivo no esta entre las caracteristicas",
                         self.objetivo not in caracteristicas, self.objetivo)

        if self.grupo and "particion" in datos.columns:
            tr = set(datos.loc[datos.particion == "train", self.grupo])
            te = set(datos.loc[datos.particion == "test", self.grupo])
            self._anotar("Sin grupos compartidos entre particiones", not (tr & te),
                         f"{len(tr)} train / {len(te)} test")

        if self.filas_esperadas is not None:
            self._anotar("Filas coinciden con lo declarado",
                         len(datos) == self.filas_esperadas,
                         f"{len(datos)} de {self.filas_esperadas}")
        return self.tabla()

    def tabla(self):
        return pd.DataFrame(self._resultados)

    @property
    def aprobado(self):
        return bool(self._resultados) and all(r["resultado"] == "OK"
                                              for r in self._resultados)

In [44]:
def probar_caso_normal():
    """El flujo completo sobre el conjunto real."""
    salida = Pipeline([ImputadorMixto(CATEGORICAS_IMPUTAR, NUMERICAS_IMPUTAR)],
                      nombre="normal").ajustar_transformar(df)
    assert len(salida) == len(df), "El pipeline perdió o duplicó filas"
    print("[OK] Caso normal: el pipeline conserva las filas y no deja nulos tratados")

    permisos_ok = agregador.transformar(limpio_f2)
    assert len(permisos_ok) == limpio_f2[GRUPO_VAR].nunique()
    print("[OK] Caso normal: la agregación devuelve un permiso por fila")

    cod = CodificadorNominal("tipo_permiso", "tipo").ajustar(df)
    salida_cod = cod.transformar(df)
    suma = salida_cod[cod.columnas_creadas].sum(axis=1)
    assert bool((suma == 1).all()), "Las indicadoras deben sumar exactamente 1"
    print("[OK] Caso normal: cada fila activa exactamente una indicadora")


def probar_casos_limite():
    """Situaciones extremas pero válidas."""
    una_fila = df.head(1)
    salida = ImputadorMixto([], NUMERICAS_IMPUTAR).ajustar_transformar(una_fila)
    assert len(salida) == 1
    print("[OK] Límite: conjunto de una sola fila")

    sin_nulos = pd.DataFrame({"x": [1.0, 2.0, 3.0]})
    antes = sin_nulos.x.tolist()
    despues = ImputadorMixto([], ["x"]).ajustar_transformar(sin_nulos).x.tolist()
    assert antes == despues, "Imputar una columna sin nulos no debe alterar valores"
    print("[OK] Límite: columna sin nulos, valores intactos")

    un_nivel = pd.DataFrame({"c": ["unico"] * 5})
    cod = CodificadorNominal("c", "c").ajustar_transformar(un_nivel)
    assert cod.shape[1] == 2 and int(cod["c_unico"].sum()) == 5
    print("[OK] Límite: variable nominal con un solo nivel")

    constante = pd.DataFrame({"z": [7.0] * 10})
    escalado = EscaladorEstandar(["z"]).ajustar_transformar(constante)
    assert bool((escalado.z == 0).all()), "Desviación cero debe dar ceros, no NaN"
    print("[OK] Límite: columna de varianza cero no produce división por cero")

    categoria_nueva = pd.DataFrame({"c": ["a", "b", "z"]})
    cod = CodificadorNominal("c", "c").ajustar(pd.DataFrame({"c": ["a", "b"]}))
    salida = cod.transformar(categoria_nueva)
    assert "c_z" not in salida.columns
    assert int(salida.loc[2, ["c_a", "c_b"]].sum()) == 0
    print("[OK] Límite: categoría no vista en ajuste no genera columna y queda en ceros")


def probar_excepciones():
    """Entradas que deben fallar, y fallar bien."""
    try:
        Transformador(["a"]).ajustar(pd.DataFrame({"a": [1]}))
        raise AssertionError("debió lanzar NotImplementedError")
    except NotImplementedError as error:
        print(f"[OK] NotImplementedError : {str(error)[:58]}")

    try:
        ImputadorMixto([], ["no_existe"]).ajustar(df)
        raise AssertionError("debió lanzar KeyError")
    except KeyError as error:
        print(f"[OK] KeyError            : {str(error)[:58]}")

    try:
        CodificadorNominal("tipo_permiso", "tipo").transformar(df)
        raise AssertionError("debió lanzar RuntimeError")
    except RuntimeError as error:
        print(f"[OK] RuntimeError        : {str(error)[:58]}")

    try:
        Pipeline().agregar("no soy un transformador")
        raise AssertionError("debió lanzar TypeError")
    except TypeError as error:
        print(f"[OK] TypeError           : {str(error)[:58]}")

    try:
        AgregadorPermiso(grupo=GRUPO_VAR,
                         constantes=["analysis_neighborhood"]).ajustar_transformar(limpio_f2)
        raise AssertionError("debió lanzar ValueError")
    except ValueError as error:
        print(f"[OK] ValueError          : {str(error)[:58]}")

    try:
        comparar_implementaciones({"sola": lambda: 1})
        raise AssertionError("debió lanzar ValueError")
    except ValueError as error:
        print(f"[OK] ValueError          : {str(error)[:58]}")

    try:
        Pipeline([ImputadorMixto([], NUMERICAS_IMPUTAR)]).ajustar(df, es_train.head(10))
        raise AssertionError("debió lanzar ValueError")
    except ValueError as error:
        print(f"[OK] ValueError          : {str(error)[:58]}")


def probar_todo():
    print("--- CASOS NORMALES ---");  probar_caso_normal()
    print("\n--- CASOS LÍMITE ---");  probar_casos_limite()
    print("\n--- EXCEPCIONES ---");   probar_excepciones()
    print("\n[OK] Batería de pruebas de la Fase 3 superada")


_, ANEXO_G3 = capturar(probar_todo)

--- CASOS NORMALES ---
[OK] Caso normal: el pipeline conserva las filas y no deja nulos tratados
[OK] Caso normal: la agregación devuelve un permiso por fila
[OK] Caso normal: cada fila activa exactamente una indicadora

--- CASOS LÍMITE ---
[OK] Límite: conjunto de una sola fila
[OK] Límite: columna sin nulos, valores intactos
[OK] Límite: variable nominal con un solo nivel
[OK] Límite: columna de varianza cero no produce división por cero
[OK] Límite: categoría no vista en ajuste no genera columna y queda en ceros

--- EXCEPCIONES ---
[OK] NotImplementedError : Cada clase hija debe implementar aprender().
[OK] KeyError            : "ImputadorMixto(no_existe): columnas ausentes: ['no_existe
[OK] RuntimeError        : CodificadorNominal(tipo_permiso): hay que ajustar antes de
[OK] TypeError           : pipeline: se esperaba un Transformador y llego str.
[OK] ValueError          : AgregadorPermiso(permit_number): estas columnas no son con
[OK] ValueError          : Se necesitan al menos

### 16.1 El conjunto final, validado

El `Validador` vive fuera del `Pipeline` porque responde otra pregunta: el `Pipeline` dice
cómo se construye el conjunto, el `Validador` dice si sirve.

Una de sus reglas cambió respecto de la Fase 2, y el motivo es consecuencia directa del
apartado 9. Allí la comprobación exigía que cada grupo de indicadoras sumara exactamente 1 en
todas las filas. Con el vocabulario aprendido solo del entrenamiento eso ya no puede
cumplirse: una fila de prueba cuya categoría nunca apareció en entrenamiento tiene **todas**
sus indicadoras en cero, y eso es lo correcto. Exigir siempre 1 equivaldría a exigir que el
codificador hubiera visto el conjunto de prueba.

La regla pasa entonces a ser *«la suma es 0 o 1»*, y el número de filas en cero se informa,
porque es una cifra que interesa: mide cuánta prueba cae fuera del vocabulario aprendido.

In [45]:
validador = Validador(objetivo=OBJETIVO, grupo=GRUPO_VAR,
                      prohibidas=PROHIBIDAS, grupos_one_hot=ONE_HOT_LIMPIO,
                      filas_esperadas=len(datos_corregidos))
tabla_validacion = validador.validar(
    datos_corregidos[CARACTERISTICAS_LIMPIAS + [OBJETIVO, "particion", GRUPO_VAR]],
    caracteristicas=CARACTERISTICAS_LIMPIAS)
print(tabla_validacion.to_string(index=False))
print(f"\nComprobaciones: {len(tabla_validacion)}  |  "
      f"aprobado: {validador.aprobado}")
if not validador.aprobado:
    raise RuntimeError("El conjunto no supera la validación: no se persiste nada.")

                                   comprobacion resultado                                                       detalle
                              Sin valores nulos        OK                                                     total = 0
        Todas las caracteristicas son numericas        OK                                              no numericas: []
                          Sin valores infinitos        OK                                                 infinitos = 0
         Grupo one-hot 'tipo_permiso' coherente        OK         10 columnas; 2 filas con categoria no vista en ajuste
               Grupo one-hot 'agente' coherente        OK         13 columnas; 0 filas con categoria no vista en ajuste
             Grupo one-hot 'distrito' coherente        OK         12 columnas; 0 filas con categoria no vista en ajuste
            Grupo one-hot 'inspector' coherente        OK         14 columnas; 0 filas con categoria no vista en ajuste
Grupo one-hot 'analysis_neighborhood' co

## 17. Documentación de la arquitectura

La tabla no se escribe a mano: se genera desde las clases realmente definidas, usando
`__subclasses__()`, que es la misma información con la que Python resuelve la herencia. Si
alguien agrega una clase y olvida documentarla, la tabla lo muestra.

In [46]:
ARCHIVO_DE = {
    "Transformador": "src/transformadores.py", "ConversorTipos": "src/transformadores.py",
    "DerivadorVariables": "src/transformadores.py", "SaneadorConjunto": "src/transformadores.py",
    "DerivadorAusencia": "src/transformadores.py", "ImputadorMixto": "src/transformadores.py",
    "CodificadorNominal": "src/transformadores.py", "EscaladorEstandar": "src/transformadores.py",
    "AgregadorPermiso": "src/transformadores.py",
    "ImputadorPorMedia": "cuaderno §14.1 (demostración de acoplamiento)",
    "Pipeline": "src/pipeline.py", "Validador": "src/pipeline.py",
    "EstrategiaImputacion": "src/transformadores.py",
}


def documentar_arquitectura():
    """Construye la tabla de arquitectura leyendo las clases definidas."""
    filas = [{"componente": "Transformador", "rol": "clase base",
              "responsabilidad": "Define el contrato y controla el estado de cada paso",
              "archivo": ARCHIVO_DE["Transformador"]}]
    for clase in Transformador.__subclasses__():
        if clase.__name__.startswith("_"):
            continue
        filas.append({"componente": clase.__name__, "rol": "clase hija",
                      "responsabilidad": (clase.__doc__ or "sin documentar").strip().split("\n")[0],
                      "archivo": ARCHIVO_DE.get(clase.__name__, "src/transformadores.py")})
    for clase in EstrategiaImputacion.__subclasses__():
        filas.append({"componente": clase.__name__, "rol": "estrategia (Strategy)",
                      "responsabilidad": f"Rellena por {clase.nombre}",
                      "archivo": "src/transformadores.py"})
    filas += [
        {"componente": "Pipeline", "rol": "orquestador",
         "responsabilidad": "Encadena los pasos y controla la máscara de ajuste",
         "archivo": "src/pipeline.py"},
        {"componente": "Validador", "rol": "verificador",
         "responsabilidad": "Comprueba integridad, coherencia y ausencia de fuga",
         "archivo": "src/pipeline.py"},
        {"componente": "medir / cronometrar", "rol": "utilidad",
         "responsabilidad": "Mide tiempo y memoria de cualquier función",
         "archivo": "src/medicion.py"},
        {"componente": "comparar_implementaciones", "rol": "utilidad",
         "responsabilidad": "Compara implementaciones exigiendo equivalencia previa",
         "archivo": "src/medicion.py"},
    ]
    return pd.DataFrame(filas)


arquitectura = documentar_arquitectura()
print(arquitectura.to_string(index=False))
sin_doc = int((arquitectura.responsabilidad == "sin documentar").sum())
print(f"\nComponentes: {len(arquitectura)}  |  sin documentar: {sin_doc}")
assert sin_doc == 0, "Hay componentes sin docstring."

               componente                   rol                                                              responsabilidad                                       archivo
            Transformador            clase base                         Define el contrato y controla el estado de cada paso                        src/transformadores.py
           ConversorTipos            clase hija            Convierte coordenadas, fechas y centinelas con formato declarado.                        src/transformadores.py
       DerivadorVariables            clase hija        Deriva el objetivo y las explicativas, y agrupa las nominales largas.                        src/transformadores.py
         SaneadorConjunto            clase hija         Descarta columnas sin valor analitico y filas sin variable objetivo.                        src/transformadores.py
        DerivadorAusencia            clase hija         Codifica la ausencia informativa como variable, en vez de imputarla.                     

### 17.1 Los módulos, generados desde este cuaderno

Las clases se definieron arriba, en celdas marcadas con `# >>> src/<archivo>.py`. Esta celda
lee **el propio cuaderno**, reúne esas celdas por archivo y escribe los módulos. De ese modo
el código de `src/` y el del cuaderno no pueden divergir: son literalmente el mismo texto.

Es el mismo principio de la Fase 1, que generaba `.gitignore` y `requirements.txt` desde el
código en vez de mantenerlos a mano.

In [47]:
CABECERAS = {
    "transformadores": ('"""Núcleo algorítmico del proyecto MCDI500 - Grupo 5, Fase 3.\n\n'
                        "Generado por F3/S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb, apartado 17.\n"
                        'No editar a mano: los cambios se hacen en el cuaderno.\n"""\n\n'
                        "import numpy as np\nimport pandas as pd\n\nimport utilidades\n"),
    "pipeline":        ('"""Orquestación y validación del núcleo algorítmico - Grupo 5, Fase 3.\n\n'
                        "Generado por F3/S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb, apartado 17.\n"
                        'No editar a mano: los cambios se hacen en el cuaderno.\n"""\n\n'
                        "import numpy as np\nimport pandas as pd\n\n"
                        "from transformadores import Transformador\n"),
    "medicion":        ('"""Medición de eficiencia - Grupo 5, Fase 3.\n\n'
                        "Generado por F3/S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb, apartado 17.\n"
                        'No editar a mano: los cambios se hacen en el cuaderno.\n"""\n\n'
                        "import math\nimport time\nimport timeit\nimport tracemalloc\n\nimport numpy as np\nimport pandas as pd\n"),
}

candidatos = sorted(DIR_F3.glob("S2_F3_*.ipynb"))
if not candidatos:
    raise FileNotFoundError(f"No se encontró el cuaderno de la Fase 3 en {DIR_F3}.")
RUTA_CUADERNO = candidatos[0]

cuaderno = json.loads(RUTA_CUADERNO.read_text(encoding="utf-8"))
bloques = {nombre: [] for nombre in CABECERAS}
for celda in cuaderno["cells"]:
    if celda["cell_type"] != "code":
        continue
    fuente = "".join(celda["source"])
    marca = fuente.split("\n", 1)[0]
    if marca.startswith("# >>> src/"):
        archivo = marca.removeprefix("# >>> src/").removesuffix(".py").strip()
        if archivo in bloques:
            bloques[archivo].append(fuente.split("\n", 1)[1].strip())

for archivo, partes in bloques.items():
    if not partes:
        raise RuntimeError(f"No se recogió ninguna celda para src/{archivo}.py")
    texto = CABECERAS[archivo] + "\n\n" + "\n\n\n".join(partes) + "\n"
    (DIR_SRC / f"{archivo}.py").write_text(texto, encoding="utf-8")
    print(f"  src/{archivo}.py  {len(partes)} bloques, {len(texto):,} caracteres")

  src/transformadores.py  8 bloques, 17,486 caracteres
  src/pipeline.py  2 bloques, 7,498 caracteres
  src/medicion.py  2 bloques, 5,142 caracteres


### 17.2 La verificación de doble vía

La Fase 2 comprobó que el módulo y el cuaderno producían el mismo conjunto. Aquí se repite
con las clases: se importan los módulos recién escritos **en un espacio de nombres limpio**,
se arma el mismo pipeline y se compara el resultado.

Si el texto de `src/` se desviara de las clases del cuaderno, esta celda lo detendría.

In [48]:
import importlib

for nombre in ("transformadores", "pipeline", "medicion"):
    if nombre in sys.modules:
        del sys.modules[nombre]

mod_trans = importlib.import_module("transformadores")
mod_pipe = importlib.import_module("pipeline")
mod_med = importlib.import_module("medicion")

pipeline_modulo = mod_pipe.Pipeline(
    [mod_trans.DerivadorAusencia(),
     mod_trans.ImputadorMixto(CATEGORICAS_IMPUTAR, NUMERICAS_IMPUTAR)]
    + [mod_trans.CodificadorNominal(c, PREFIJOS[c]) for c in NOMINALES]
    + [mod_trans.EscaladorEstandar(NUMERICAS)], nombre="desde_modulo")

salida_modulo = pipeline_modulo.ajustar(partida, mascara).transformar(partida)
salida_modulo[OBJETIVO_LOG] = np.log1p(salida_modulo[OBJETIVO])

columnas = CARACTERISTICAS_LIMPIAS + [OBJETIVO, OBJETIVO_LOG, "particion", GRUPO_VAR]
pd.testing.assert_frame_equal(
    salida_modulo[columnas].reset_index(drop=True),
    datos_corregidos[columnas].reset_index(drop=True))
print("*** El pipeline del módulo y el del cuaderno producen el MISMO conjunto ***")

agregador_modulo = mod_trans.AgregadorPermiso(
    grupo=GRUPO_VAR, constantes=COLUMNAS_PERMISO,
    modales=COLUMNAS_MODALES, medias=COLUMNAS_MEDIAS)
pd.testing.assert_frame_equal(agregador_modulo.ajustar_transformar(limpio_f2), permisos)
print("*** El AgregadorPermiso del módulo y el del cuaderno coinciden ***")
print(f"\nClases disponibles en src/transformadores.py: "
      f"{len(mod_trans.Transformador.__subclasses__())} hijas de Transformador")

*** El pipeline del módulo y el del cuaderno producen el MISMO conjunto ***


*** El AgregadorPermiso del módulo y el del cuaderno coinciden ***

Clases disponibles en src/transformadores.py: 8 hijas de Transformador
